# Tablero de Carnes


## 0. Setup

In [ ]:
# Entorno local: no requiere montar Google Drive.
from entorno_local import archivo_config, ruta_local
from IPython.display import display
# Opcional: configurar os.environ["TABLERO_CONFIG"],
# os.environ["TABLERO_INPUT_DIR"] y os.environ["TABLERO_OUTPUT_DIR"].


In [ ]:
import pandas as pd
import numpy as np
import os, re, warnings
warnings.filterwarnings('ignore')

print('Setup OK — imports listos. INPUT_DIR, OUTPUT_DIR y el resto de los parámetros')
print('se cargan en la celda siguiente desde config_parametros_tablero.xlsx.')


## 0.1 Parámetros (desde Excel)
Se leen una sola vez de `config_parametros_tablero.xlsx`. Para cambiar un umbral, un mínimo o una ruta, se edita ESE archivo — no esta celda.

In [ ]:
# ============ 0.1 · Parámetros desde Excel (config_parametros_tablero.xlsx) ============
# Todo lo que antes vivía hardcodeado en el código (umbrales del semáforo, mínimos usados
# en los cálculos, carpetas de entrada/salida, meses a procesar) se lee acá de un único
# Excel. Para cambiar un parámetro se edita esa celda del Excel, no esta celda de código.
#
# Esta celda también UNIFICA algo que antes no lo estaba: los umbrales de acá alimentan
# tanto el Excel de salida (antes celda "Semáforo por Grupo") como el JSON que lee el HTML
# (antes celda "UMBRALES_META" más abajo), que habían quedado desalineados entre sí para
# varios indicadores. Ver la columna "Nota" en la hoja Indicadores_Semaforo del Excel.

# Poné acá la carpeta INPUT (o directamente la ruta completa al archivo, las dos formas andan)
CONFIG_XLSX = archivo_config()

def _leer_hoja_config(hoja, **kw):
    try:
        return pd.read_excel(CONFIG_XLSX, sheet_name=hoja, **kw)
    except Exception as e:
        raise RuntimeError(
            f'No pude leer la hoja "{hoja}" de {os.path.basename(CONFIG_XLSX)}. '
            f'¿Existe el archivo en esa carpeta y tiene esa hoja? Detalle: {e}'
        )

_df_ind   = _leer_hoja_config('Indicadores_Semaforo')
_df_grp   = _leer_hoja_config('Grupos')
_df_gen   = _leer_hoja_config('Parametros_Generales')
_df_rutas = _leer_hoja_config('Rutas')
_df_meses = _leer_hoja_config('Meses_Procesar')

_ESPERADOS = {'01','03','04','05','06','07','08','09','10','11','12','13','14','15','16'}
_cargados  = set(_df_ind['Codigo'].astype(str).str.zfill(2))
if _ESPERADOS - _cargados:
    raise ValueError(f'Faltan indicadores en la hoja Indicadores_Semaforo del Excel de parámetros: {sorted(_ESPERADOS - _cargados)}')
if _cargados - _ESPERADOS:
    print(f'⚠️  Hay códigos de indicador en el Excel que el notebook no reconoce: {sorted(_cargados - _ESPERADOS)}')

_nombre_grupo = dict(zip(_df_grp['Grupo'].astype(str), _df_grp['Nombre'].astype(str)))

def _es_si(v):
    return str(v).strip().lower() in ('si', 'sí', 'true', '1', '1.0')

UMBRALES, UMBRALES_META, EXCLUIR_SI_GRIS = {}, {}, set()
GRUPOS = {}
for _, r in _df_ind.iterrows():
    cod  = str(r['Codigo']).zfill(2)
    tipo = str(r['Direccion']).strip().lower()
    amar = None if pd.isna(r['Amarillo']) else float(r['Amarillo'])
    rojo = None if pd.isna(r['Rojo']) else float(r['Rojo'])
    grp  = str(r['Grupo']).strip()

    UMBRALES[cod] = (tipo, amar, rojo)

    meta = {'nombre': str(r['Nombre']), 'dir': tipo, 'binario': _es_si(r['Binario']), 'grupo': grp}
    if amar is not None: meta['amarillo'] = amar
    if rojo is not None: meta['rojo'] = rojo
    UMBRALES_META[cod] = meta

    nombre_grupo_completo = f"{grp} · {_nombre_grupo.get(grp, grp)}"
    GRUPOS.setdefault(nombre_grupo_completo, []).append(cod)

    if _es_si(r['ExcluirSiGris']):
        EXCLUIR_SI_GRIS.add(cod)

GRUPOS_META = {}
for _, r in _df_grp.iterrows():
    g = str(r['Grupo']).strip()
    codigos_g = [c for c, meta in UMBRALES_META.items() if meta['grupo'] == g]
    GRUPOS_META[g] = {'nombre': str(r['Nombre']), 'indicadores': codigos_g}

PARAMS = dict(zip(_df_gen['Parametro'].astype(str), _df_gen['Valor']))
MIN_ANIMALES_TROPA     = int(PARAMS['MIN_ANIMALES_TROPA'])
VENTANA_DIAS            = int(PARAMS['VENTANA_DIAS'])
DEMORA_FAENA_DIAS       = int(PARAMS['DEMORA_FAENA_DIAS'])
HORA_LIMITE_APROBACION  = str(PARAMS['HORA_LIMITE_APROBACION'])
PUNTOS = {
    'ROJO':     float(PARAMS['PUNTOS_ROJO']),
    'AMARILLO': float(PARAMS['PUNTOS_AMARILLO']),
    'VERDE':    float(PARAMS['PUNTOS_VERDE']),
    'GRIS':     float(PARAMS['PUNTOS_GRIS']),
}
PUNTOS_VIOLETA       = float(PARAMS['PUNTOS_VIOLETA'])
TAMANIO_CHICO_MAX    = float(PARAMS['TAMANIO_CHICO_MAX'])
TAMANIO_MEDIANO_MAX  = float(PARAMS['TAMANIO_MEDIANO_MAX'])
PADRON_TOTAL         = int(PARAMS['PADRON_TOTAL'])   # lo usa Notebook 2

RUTAS = dict(zip(_df_rutas['Parametro'].astype(str), _df_rutas['Valor']))
INPUT_DIR           = ruta_local(RUTAS['INPUT_DIR'], 'INPUT_DIR')
OUTPUT_DIR          = ruta_local(RUTAS['OUTPUT_DIR'], 'OUTPUT_DIR')
NOMBRE_EXCEL_SALIDA = RUTAS['NOMBRE_EXCEL_SALIDA']
NOMBRE_JSON_SALIDA  = RUTAS['NOMBRE_JSON_SALIDA']
os.makedirs(OUTPUT_DIR, exist_ok=True)

MESES = list(zip(_df_meses['Anio'].astype(int), _df_meses['Mes'].astype(int)))

print(f'✅ Parámetros cargados desde {os.path.basename(CONFIG_XLSX)}')
print(f'   Indicadores: {len(UMBRALES)} | Grupos: {list(GRUPOS.keys())}')
print(f'   Meses a procesar: {MESES}')
print(f'   INPUT_DIR : {INPUT_DIR}')
print(f'   OUTPUT_DIR: {OUTPUT_DIR}')

## 1. Detección automática de archivos

In [ ]:
BASE_KEYWORDS = {
    'autorizaciones':            ['autorizaciones', 'autorizacion', 'vep'],
    'capacidad_camaras':         ['capacidadcamaras', 'capacidad_camaras', 'camaras', 'capacidad', 'corral'],
    'existencias':               ['existencias', 'existencia', 'stock'],
    'faena':                     ['faena', 'romaneo'],
    'ingresos':                  ['ingresos', 'ingreso'],
    'tipificacion':              ['tipificacion', 'tipificación'],
    'tipificadores_habilitados': ['tipificadoreshabilitados', 'tipificadores_habilitados', 'tipificadores', 'habilitados', 'padron'],
    'sigsa':                     ['sigsa','movimiento','movimientos'],
    'padron_siocal':             ['padron', 'padrón', 'listado', 'siocal'],
    'padron_sigica':             ['sigica', 'padronsigica'],
    'scoring': ['scoring', 'inspecciones', 'inspeccion'],
    'habilitacion_faena':        ['habilitacion', 'mataderos', 'cupo'],
}
BASE_ORDER = [
    'tipificadores_habilitados', 'capacidad_camaras', 'autorizaciones',
    'existencias', 'faena', 'ingresos', 'tipificacion', 'sigsa',
    'padron_sigica', 'padron_siocal','habilitacion_faena',
]
print(f'{len(BASE_KEYWORDS)} bases configuradas')


In [ ]:
def _slug(s):
    s = s.lower()
    for a, b in [('á','a'),('é','e'),('í','i'),('ó','o'),('ú','u'),('ñ','n')]:
        s = s.replace(a, b)
    return re.sub(r'[^a-z0-9]', '', s)

def detectar_bases(input_dir, keywords=BASE_KEYWORDS, orden=BASE_ORDER):
    archivos = [f for f in os.listdir(input_dir) if f.lower().endswith(('.xlsx','.xls','.csv')) and not f.startswith('~$')]
    asignacion, sin_match = {}, []
    for archivo in archivos:
        slug_archivo = _slug(archivo)
        puntajes = {}
        for base, kws in keywords.items():
            score = max((len(_slug(kw)) for kw in kws if _slug(kw) in slug_archivo), default=0)
            if score > 0:
                puntajes[base] = score
        if not puntajes:
            sin_match.append(archivo)
            continue
        max_score = max(puntajes.values())
        candidatos = [b for b, s in puntajes.items() if s == max_score]
        ganador = candidatos[0] if len(candidatos) == 1 else next(b for b in orden if b in candidatos)
        if ganador in asignacion:
            print(f'⚠️  Conflicto: "{archivo}" y "{asignacion[ganador]}" → "{ganador}". Se queda el primero.')
            continue
        asignacion[ganador] = os.path.join(input_dir, archivo)
    return asignacion, sin_match

asignacion, sin_match = detectar_bases(INPUT_DIR)
print('Bases detectadas:\n')
for base in BASE_KEYWORDS:
    estado = f'✅ ← {os.path.basename(asignacion[base])}' if base in asignacion else '❌ NO DETECTADO'
    print(f'  {base:<28} {estado}')
if sin_match:
    print('\nSin match:', sin_match)


### 1.2 — Asignación manual (solo si hace falta)
Completá el nombre EXACTO del archivo donde haga falta. Dejá `None` en las que ya se detectaron bien.

In [ ]:
ASIGNACION_MANUAL = {
    'autorizaciones':            None,
    'capacidad_camaras':         None,
    'existencias':               None,
    'faena':                     None,
    'ingresos':                  None,
    'tipificacion':              None,
    'tipificadores_habilitados': None,
    'sigsa':                     None,
    'padron_siocal':             None,
    'padron_sigica':             None,
    'scoring': None,
    'habilitacion_faena':        None,
}
for base, filename in ASIGNACION_MANUAL.items():
    if filename:
        path = os.path.join(INPUT_DIR, filename)
        if not os.path.exists(path):
            print(f'⚠️  "{filename}" no existe en INPUT_DIR')
        else:
            asignacion[base] = path
            print(f'✅ Asignado manualmente: {base} ← {filename}')
faltantes = [b for b in BASE_KEYWORDS if b not in asignacion]
if faltantes:
    print(f'⚠️  Sin archivo: {faltantes}')
else:
    print('✅ Las 10 bases tienen archivo asignado.')


## 2. Funciones de limpieza reutilizables

In [ ]:
def fix_encoding(s):
    if not isinstance(s, str): return s
    try: return s.encode('latin-1').decode('utf-8')
    except: return s

def fix_df_encoding(df):
    df.columns = [fix_encoding(c) for c in df.columns]
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].map(fix_encoding)
    return df

def normalizar_numerico(serie, nombre_campo='', nombre_base=''):
    numerico = pd.to_numeric(serie, errors='coerce')
    out = numerico.round().astype('Int64')
    n_perdidos = out.isna().sum() - serie.isna().sum()
    if n_perdidos > 0:
        print(f'  ⚠️  {nombre_base}.{nombre_campo}: {n_perdidos} valores no numéricos → NA')
    return out

def normalizar_cuit(serie):
    def _parse(v):
        if pd.isna(v): return pd.NA
        s = re.sub(r'[^0-9]', '', str(v))
        return int(s) if len(s) == 11 else pd.NA
    return serie.map(_parse).astype('Int64')

def normalizar_fecha(serie):
    def _parse(v):
        if pd.isna(v): return pd.NaT
        if isinstance(v, (int, float)):
            try: return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(v))
            except: return pd.NaT
        return pd.to_datetime(v, errors='coerce')
    return serie.map(_parse)

def normalizar_dte(serie):
    def _parse(v):
        if pd.isna(v): return pd.NA
        s = re.sub(r'[^0-9]', '', str(v))
        return f'{s[:9]}-{s[9]}' if len(s) == 10 else str(v)
    return serie.map(_parse)

def renombrar_cols(df, mapa):
    return df.rename(columns={k: v for k, v in mapa.items() if k in df.columns})

def informe_base(nombre, df):
    print(f'\n{"="*55}')
    print(f'  {nombre}  —  {len(df):,} filas  |  {len(df.columns)} columnas')
    print(f'{"="*55}')
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print('Nulos:')
        print(nulls.to_string())
    print()

print('Funciones de limpieza OK')


## 3. Padrón SIOCAL — Base Madre
Fuente del universo de establecimientos. Una fila por planta.
Columnas que se usan: `Planta` (= establecimiento_ruca), `CUIT`, `Razon Social`, `Provincia`, `Actividad`, `Estado`, especies.

In [ ]:
# --- Lector unificado de bases: CSV o Excel indistinto, con motor rápido para xlsx ---
# calamine es ~5-20x más rápido que openpyxl en archivos grandes. Si no está instalado,
# leer_base cae automáticamente al motor default de pandas.
# python-calamine se instala desde requirements.txt.

def leer_base(path, sheet_name=0, header=0, skiprows=None, dtype=None, usecols=None):
    ext = os.path.splitext(str(path))[1].lower()
    if ext == '.csv':
        # motor C (como el original, tolerante con comillas); probamos separadores conocidos
        for _sep in (';', ',', '\t'):
            try:
                df = pd.read_csv(path, encoding='latin-1', sep=_sep,
                                 header=header, skiprows=skiprows, dtype=dtype, usecols=usecols)
                if df.shape[1] > 1:          # si dio 1 sola columna, el separador estaba mal
                    return df
            except Exception:
                pass
        # último recurso: autodetección (motor python)
        return pd.read_csv(path, encoding='latin-1', sep=None, engine='python',
                           header=header, skiprows=skiprows, dtype=dtype, usecols=usecols)
    for _engine in ('calamine', None):       # xlsx: calamine primero, fallback al default
        try:
            return pd.read_excel(path, sheet_name=sheet_name, header=header,
                                 skiprows=skiprows, dtype=dtype, usecols=usecols, engine=_engine)
        except Exception:
            if _engine is None:
                raise

In [ ]:
df_pad = leer_base(asignacion['padron_siocal'])
df_pad = fix_df_encoding(df_pad)

df_pad = renombrar_cols(df_pad, {
    'Planta':           'establecimiento_ruca',
    'CUIT':             'cuit_titular',
    'Razon Social':     'razon_social',
    'Provincia':        'provincia',
    'Actividad':        'actividad',
    'Estado':           'estado',
    'Especie Bovino':   'especie_bovino',
    'Especie Ovino':    'especie_ovino',
    'Especie Porcino':  'especie_porcino',
    'Especie Equina':   'especie_equina',
    'Especie Caprina':  'especie_caprina',
    'Especie Avicola':  'especie_avicola',
})

df_pad['establecimiento_ruca'] = normalizar_numerico(df_pad['establecimiento_ruca'], 'establecimiento_ruca', 'Padron_SIOCAL')
df_pad['cuit_titular'] = normalizar_cuit(df_pad['cuit_titular'])

# Columna especies unificada
ESPECIES_COLS = ['especie_bovino','especie_ovino','especie_porcino','especie_equina','especie_caprina','especie_avicola']
ESPECIES_NOMBRES = ['Bovino','Ovino','Porcino','Equino','Caprino','Avicola']
def armar_especies(row):
    return ', '.join([n for col, n in zip(ESPECIES_COLS, ESPECIES_NOMBRES)
                     if str(row.get(col,'')).strip().upper() == 'SI'])
df_pad['especies'] = df_pad.apply(armar_especies, axis=1)

print(f'Padrón SIOCAL: {len(df_pad)} establecimientos')
print(f'Estados: {df_pad["estado"].value_counts().to_dict()}')
df_pad[['establecimiento_ruca','razon_social','cuit_titular','provincia','actividad','especies','estado']].head(3)


## 3b. Scoring e Inspecciones
Datos traídos desde el tablero de inspecciones (snapshot). Se cruzan por `establecimiento_ruca` (= `Nro Planta`) y se agregan como columnas en la tabla maestra antes de los indicadores.

In [ ]:
# --- Scoring e inspecciones desde Google Sheets (snapshot) ---
# Fuente: hoja 'PADRON CARNES' del archivo de inspecciones
# Clave de join: Nro Planta = establecimiento_ruca
# Columnas traídas: Última Inspección (Q) y Scoring Actual (Z)

df_scoring = leer_base(
    asignacion['scoring'],
    sheet_name='PADRON CARNES',
    usecols=['Nro Planta', 'Última Inspección', 'Scoring Actual '],
)
df_scoring = fix_df_encoding(df_scoring)
df_scoring = df_scoring.rename(columns={
    'Nro Planta':        'establecimiento_ruca',
    'Última Inspección': 'ultima_inspeccion',
    'Scoring Actual ':   'scoring_actual',
})
df_scoring['establecimiento_ruca'] = normalizar_numerico(df_scoring['establecimiento_ruca'], 'establecimiento_ruca', 'Scoring')
df_scoring['ultima_inspeccion']    = normalizar_fecha(df_scoring['ultima_inspeccion'])
df_scoring = df_scoring.dropna(subset=['establecimiento_ruca']).drop_duplicates(subset='establecimiento_ruca')

print(f'Scoring: {len(df_scoring)} establecimientos')
print(f'Con última inspección: {df_scoring["ultima_inspeccion"].notna().sum()}')
print(f'Con scoring actual: {df_scoring["scoring_actual"].notna().sum()}')
df_scoring.head(3)


## 4. Padrón SIGICA — Mapeo Numero Oficial → establecimiento_ruca
Se usa para resolver el cruce en SIGSA cuando no se puede hacer directo por RUCA.

In [ ]:
df_sigica = leer_base(asignacion['padron_sigica'])
df_sigica = fix_df_encoding(df_sigica)

df_sigica = renombrar_cols(df_sigica, {
    'Numero Oficial':      'numero_oficial',
    'Establecimiento RUCA':'establecimiento_ruca',
    'CUIT Titular':        'cuit_titular',
    'Razon Social':        'razon_social',
})

df_sigica['establecimiento_ruca'] = normalizar_numerico(df_sigica['establecimiento_ruca'], 'establecimiento_ruca', 'Padron_SIGICA')
df_sigica['matricula_ruca'] = normalizar_numerico(df_sigica['Matricula RUCA'], 'matricula_ruca', 'Padron_SIGICA')
df_sigica['numero_oficial'] = df_sigica['numero_oficial'].astype(str).str.strip()
df_sigica.loc[df_sigica['numero_oficial'] == 'nan', 'numero_oficial'] = None
# Diccionario numero_oficial → establecimiento_ruca
MAP_NOFICIAL_RUCA = (
    df_sigica[df_sigica['numero_oficial'].notna() & df_sigica['establecimiento_ruca'].notna()]
    .drop_duplicates(subset='numero_oficial')
    .set_index('numero_oficial')['establecimiento_ruca']
    .to_dict()
)
print(f'Padrón SIGICA: {len(df_sigica)} registros')
print(f'Mapeo Numero Oficial → RUCA: {len(MAP_NOFICIAL_RUCA)} entradas')


## 5. Autorizaciones

In [ ]:
df_aut = leer_base(asignacion['autorizaciones'], sheet_name='Hoja1')
df_aut = fix_df_encoding(df_aut)
df_aut = renombrar_cols(df_aut, {
    'Razon Social':        'razon_social',
    'Id Tropa':            'id_tropa',
    'Numero Tropa':        'numero_tropa',
    'Fecha Faena':         'fecha_faena',
    'Numero Planilla':     'numero_planilla',
    'Especie':             'especie',
    'Eliminada':           'eliminada',
    'Intervenida':         'intervenida',
    'Cantidad Faenada':    'cantidad_faenada',
    'Cantidad Autorizada': 'cantidad_autorizada',
    'Tipo Autorizacion':   'tipo_autorizacion',
    'Tipo Emergencia':     'tipo_emergencia',
    'Establecimiento RUCA':'establecimiento_ruca',
    'CUIT Titular':        'cuit_titular',
    'Dte':                 'dte',
    'Autorizacion Hora':   'autorizacion_hora',
    'Autorizacion Fecha Aprobada': 'autorizacion_fecha_aprobada',
})
df_aut['fecha_faena']          = normalizar_fecha(df_aut['fecha_faena'])
df_aut['cuit_titular']         = normalizar_cuit(df_aut['cuit_titular'])
df_aut['dte']                  = normalizar_dte(df_aut['dte'])
df_aut['establecimiento_ruca'] = normalizar_numerico(df_aut['establecimiento_ruca'], 'establecimiento_ruca', 'Autorizaciones')
df_aut['autorizacion_fecha_aprobada'] = pd.to_datetime(df_aut['autorizacion_fecha_aprobada'], errors='coerce')
df_aut['es_emergencia'] = df_aut['tipo_emergencia'].notna()
df_aut['es_posdatada']  = df_aut['tipo_emergencia'].astype(str).str.strip() == 'Carga Posdatada'
if 'autorizacion_hora' not in df_aut.columns:
    df_aut['autorizacion_hora'] = pd.NA
    print('⚠️  autorizacion_hora no encontrada — indicador 08 vacío')
informe_base('AUTORIZACIONES', df_aut)

## 7. Existencias

In [ ]:
df_exi = leer_base(asignacion['existencias'])
df_exi = fix_df_encoding(df_exi)
df_exi = renombrar_cols(df_exi, {
    'Establecimiento RUCA': 'establecimiento_ruca',
    'CUIT Titular':         'cuit_titular',
    'Fecha Existencia':     'fecha_existencia',
    'Especie':              'especie',
    'Nro Tropa':            'numero_tropa',
    'Periodo':              'periodo',
    'Nro Camara':           'numero_camara',
    'Titular Faena':        'titular_faena',
    'Cantidad':             'cantidad',
    'Kilos':                'kilos',
    'Fecha Faena':          'fecha_faena',
    'Corte':                'corte',
    'Unidad Camara':        'unidad_camara',
    'Numero Oficial':       'numero_oficial',
})
df_exi['establecimiento_ruca'] = normalizar_numerico(df_exi['establecimiento_ruca'], 'establecimiento_ruca', 'Existencias')
df_exi['cuit_titular']         = normalizar_cuit(df_exi['cuit_titular'])
df_exi['fecha_existencia']     = normalizar_fecha(df_exi['fecha_existencia'])
df_exi['fecha_faena']          = normalizar_fecha(df_exi['fecha_faena'])
df_exi['cantidad']             = normalizar_numerico(df_exi['cantidad'], 'cantidad', 'Existencias')
if 'numero_oficial' not in df_exi.columns:
    df_exi['numero_oficial'] = pd.NA
else:
    df_exi['numero_oficial'] = normalizar_numerico(df_exi['numero_oficial'], 'numero_oficial', 'Existencias')
informe_base('EXISTENCIAS', df_exi)


## 8. Faena

In [ ]:
df_fae = leer_base(asignacion['faena'])
df_fae = fix_df_encoding(df_fae)
df_fae = renombrar_cols(df_fae, {
    'Numero Oficial':          'numero_oficial',
    'Razon Social':            'razon_social',
    'Fecha Faena':             'fecha_faena',
    'Especie':                 'especie',
    'Eliminada':               'eliminada',
    'Intervenida':             'intervenida',
    'Cantidad Detalle Corte':  'cantidad_detalle_corte',
    'Corte':                   'corte',
    'Categoria Detalle Faena': 'categoria_faena',
    'Kilos Detalle Faena':     'kilos_faena',
    'Establecimiento RUCA':    'establecimiento_ruca',
    'CUIT Titular':            'cuit_titular',
    'Dte':                     'dte',
    'Numero Tropa':            'numero_tropa',
})
df_fae['fecha_faena']           = normalizar_fecha(df_fae['fecha_faena'])
df_fae['cuit_titular']          = normalizar_cuit(df_fae['cuit_titular'])
df_fae['dte']                   = normalizar_dte(df_fae['dte'])
df_fae['numero_oficial']        = normalizar_numerico(df_fae['numero_oficial'], 'numero_oficial', 'Faena')
df_fae['establecimiento_ruca']  = normalizar_numerico(df_fae['establecimiento_ruca'], 'establecimiento_ruca', 'Faena')
df_fae['cantidad_detalle_corte']= normalizar_numerico(df_fae['cantidad_detalle_corte'], 'cantidad_detalle_corte', 'Faena')
corte_norm = df_fae['corte'].astype(str).str.strip().str.upper()
divisor = pd.Series(np.nan, index=df_fae.index)
divisor[corte_norm.isin(['1/2 RES', '1/2 RESES'])] = 2
divisor[corte_norm.isin(['1/4 DELANTERO', '1/4 DELANTEROS', '1/4 TRASERO', '1/4 TRASEROS'])] = 4
divisor[corte_norm.isin(['CARCASA', 'ENTERO / CARCASA'])] = 1
sin_mapear = df_fae.loc[divisor.isna(), 'corte'].value_counts()
if not sin_mapear.empty:
    print('⚠️  Valores de corte sin divisor (quedan NA en cantidad_cabezas):')
    print(sin_mapear.to_string())
df_fae['cantidad_cabezas'] = df_fae['cantidad_detalle_corte'] / divisor
informe_base('FAENA', df_fae)


## 9. Ingresos
Ahora incluye `Cuit Usuario Faena` y `Nombre Usuario Faena`. Esta base es el puente para derivar el usuario en los indicadores que no tienen ese campo directamente.

In [ ]:
df_ing = leer_base(asignacion['ingresos'])
df_ing = fix_df_encoding(df_ing)
df_ing = renombrar_cols(df_ing, {
    'Numero Oficial':       'numero_oficial',
    'Razon Social':         'razon_social',
    'Fecha Ingreso':        'fecha_ingreso',
    'Eliminada':            'eliminada',
    'Intervenida':          'intervenida',
    'Especie':              'especie',
    'Cantidad Ingresada':   'cantidad_ingresada',
    'Dte':                  'dte',
    'Matricula RUCA':       'matricula_ruca',
    'Establecimiento RUCA': 'establecimiento_ruca',
    'CUIT Titular':         'cuit_titular',
    'Cuit Usuario Faena':   'cuit_usuario_faena',
    'Nombre Usuario Faena': 'nombre_usuario_faena',
})
df_ing['fecha_ingreso']        = normalizar_fecha(df_ing['fecha_ingreso'])
df_ing['dte']                  = normalizar_dte(df_ing['dte'])
df_ing['numero_oficial']       = normalizar_numerico(df_ing['numero_oficial'], 'numero_oficial', 'Ingresos')
df_ing['establecimiento_ruca'] = normalizar_numerico(df_ing['establecimiento_ruca'], 'establecimiento_ruca', 'Ingresos')
df_ing['cantidad_ingresada']   = normalizar_numerico(df_ing['cantidad_ingresada'], 'cantidad_ingresada', 'Ingresos')
df_ing['cuit_titular']         = normalizar_cuit(df_ing['cuit_titular'])
df_ing['cuit_usuario_faena']   = normalizar_cuit(df_ing['cuit_usuario_faena'])

# Tabla de mapeo DTE → usuario (para cruzar con otras bases)
DTE_USUARIO = (
    df_ing[['dte', 'establecimiento_ruca', 'cuit_usuario_faena', 'nombre_usuario_faena']]
    .drop_duplicates(subset='dte')
)
print(f'Ingresos: {len(df_ing):,} filas')
print(f'DTe únicos: {df_ing["dte"].nunique():,}')
print(f'Usuarios únicos: {df_ing["cuit_usuario_faena"].nunique():,}')
informe_base('INGRESOS', df_ing)


## 10. Tipificación

In [ ]:
df_tip = leer_base(asignacion['tipificacion'], dtype={'Numero Oficial': str})
df_tip = fix_df_encoding(df_tip)
df_tip = renombrar_cols(df_tip, {
    'Numero Oficial':            'numero_oficial',
    'Razon Social':              'razon_social',
    'Fecha Faena':               'fecha_faena',
    'Especie':                   'especie',
    'Eliminada':                 'eliminada',
    'Numero Tipificador':        'numero_tipificador',
    'Categoria Tipificacion':    'categoria_tipificacion',
    'Destino Tipificacion':      'destino_tipificacion',
    'Tipo Tipificacion':         'tipo_tipificacion',
    'Kilos Tipificacion':        'kilos_tipificacion',
    'Cantidad Tipificacion':     'cantidad_tipificacion',
    'Matricula RUCA':            'matricula_ruca',
    'Establecimiento RUCA':      'establecimiento_ruca',
    'CUIT Titular':              'cuit_titular',
    'NewColumn.Categoría simple':'categoria_simple',
    'Numero Tropa':              'numero_tropa',
})

# --- Dentición: se calcula acá a partir de categoria_tipificacion ---
# (antes venía pre-armada a mano en Excel; con el volumen de filas actual dejó de ser práctico)
MAP_DENTICION = {
    '>8D - NO': 'Más de 8 dientes', '>8D - TO': 'Más de 8 dientes', '>8D - VA': 'Más de 8 dientes',
    '2D - MEJ': '2 dientes',        '2D - NT':  '2 dientes',        '2D - VQ':  '2 dientes',
    '4D - NT':  '4 dientes',        '4D - TO':  '4 dientes',        '4D - VQ':  '4 dientes',
    '6D - NO':  '6 dientes',        '6D - TO':  '6 dientes',        '6D - VA':  '6 dientes',
    '8D - NO':  '8 dientes',        '8D - TO':  '8 dientes',        '8D - VA':  '8 dientes',
}
df_tip['denticion'] = df_tip['categoria_tipificacion'].astype(str).str.strip().map(MAP_DENTICION)

_sin_mapear_dent = df_tip.loc[df_tip['denticion'].isna() & df_tip['categoria_tipificacion'].notna(), 'categoria_tipificacion'].value_counts()
if not _sin_mapear_dent.empty:
    print('⚠️  Categorías de Tipificación sin Dentición mapeada (quedan NA):')
    print(_sin_mapear_dent.to_string())

df_tip['fecha_faena']          = normalizar_fecha(df_tip['fecha_faena'])

# --- numero_mes: ya no viene en la base; se calcula como el mes (1-12) de fecha_faena ---
df_tip['numero_mes'] = df_tip['fecha_faena'].dt.month.astype('Int64')

df_tip['cuit_titular']         = normalizar_cuit(df_tip['cuit_titular'])
df_tip['numero_oficial']       = normalizar_numerico(df_tip['numero_oficial'], 'numero_oficial', 'Tipificacion')
df_tip['establecimiento_ruca'] = normalizar_numerico(df_tip['establecimiento_ruca'], 'establecimiento_ruca', 'Tipificacion')
df_tip['numero_tipificador']   = normalizar_numerico(df_tip['numero_tipificador'], 'numero_tipificador', 'Tipificacion')
TIENE_NUMERO_TROPA = 'numero_tropa' in df_tip.columns
if TIENE_NUMERO_TROPA:
    df_tip['numero_tropa'] = normalizar_numerico(df_tip['numero_tropa'], 'numero_tropa', 'Tipificacion')
    print('✅ numero_tropa disponible — indicadores 10 habilitado')
else:
    print('⚠️  numero_tropa NO disponible — indicador 10 deshabilitado')
informe_base('TIPIFICACIÓN', df_tip)


In [ ]:
print(df_tip.columns.tolist())
print(df_tip.head(3))

## 11. Tipificadores Habilitados

In [ ]:
df_hab = leer_base(asignacion['tipificadores_habilitados'])
df_hab = fix_df_encoding(df_hab)
cols_drop = ['Capacidad de Almacenaje','Capacidad de Molienda','Volumen de Recibo',
             'Coordenadas','Provincia','Departamento','Localidad','Dirección']
df_hab = df_hab.drop(columns=[c for c in cols_drop if c in df_hab.columns])
df_hab = renombrar_cols(df_hab, {
    'Nro Tipificador':  'nro_tipificador',
    'Nro Inscripción':  'nro_inscripcion',
    'CUIT':             'cuit_tipificador',
    'Razon Social':     'razon_social',
    'Matricula':        'matricula',
    'Actividad':        'actividad',
    'Código Actividad': 'codigo_actividad',
    'Rubro':            'rubro',
    'Inicio':           'fecha_inicio',
    'Vencimiento':      'fecha_vencimiento',
    'Estado':           'estado',
    'Planta':           'establecimiento_ruca',
    'Analista':         'analista',
    'Especie Bovino':   'especie_bovino',
    'Especie Ovino':    'especie_ovino',
    'Especie Porcino':  'especie_porcino',
    'Especie Equina':   'especie_equina',
    'Especie Caprina':  'especie_caprina',
    'Especie Avicola':  'especie_avicola',
    'Tipo Tipificador': 'tipo_tipificador',
})
df_hab['cuit_tipificador']     = normalizar_cuit(df_hab['cuit_tipificador'])
df_hab['establecimiento_ruca'] = normalizar_numerico(df_hab['establecimiento_ruca'], 'establecimiento_ruca', 'Tipificadores_Habilitados')
df_hab['nro_tipificador']      = normalizar_numerico(df_hab['nro_tipificador'], 'nro_tipificador', 'Tipificadores_Habilitados')
df_hab['fecha_inicio']         = normalizar_fecha(df_hab['fecha_inicio'])
df_hab['fecha_vencimiento']    = normalizar_fecha(df_hab['fecha_vencimiento'])
df_hab['habilitado']           = df_hab['estado'].astype(str).str.upper().str.strip() == 'VIGENTE'
informe_base('TIPIFICADORES HABILITADOS', df_hab)


## 12. SIGSA
**Claves:** `establecimiento_ruca` (campo `RUCA`). Si algún registro no tiene RUCA, se resuelve
vía `Número Oficial` → `MAP_NOFICIAL_RUCA` (padrón SIGICA).

In [ ]:
df_sig = leer_base(asignacion['sigsa'])
df_sig = fix_df_encoding(df_sig)
df_sig = renombrar_cols(df_sig, {
    'DTe':                          'dte',
    'Fecha de Movimiento (Fecha)':  'fecha_movimiento',
    'Mes de Movimiento':            'mes_movimiento',
    'Año de Movimiento':            'anio_movimiento',
    'Fecha de Emisión DTe (Fecha)': 'fecha_emision_dte',
    'Fecha de Vencimiento (Fecha)': 'fecha_vencimiento_dte',
    'Motivo':                       'motivo',
    'Estado':                       'estado',
    'Especie':                      'especie',
    'Tipo origen':                  'tipo_origen',
    'Tipo destino':                 'tipo_destino',
    'Categoría':                    'categoria',
    'Cantidad':                     'cantidad_amparada',
    'Cantidad al cierre':           'cantidad_cierre',
    'Codigo Frigorifico':           'codigo_frigorifico',
    'Nombre Frigorifico':           'nombre_frigorifico',
    'Nombre Provincia Frigorifico': 'provincia_frigorifico',
    'Número Oficial':               'numero_oficial',
    'RUCA':                         'matricula_ruca',
})
df_sig['dte']                   = normalizar_dte(df_sig['dte'])
df_sig['fecha_movimiento']      = normalizar_fecha(df_sig['fecha_movimiento'])
df_sig['fecha_emision_dte']     = normalizar_fecha(df_sig['fecha_emision_dte'])
df_sig['fecha_vencimiento_dte'] = normalizar_fecha(df_sig['fecha_vencimiento_dte'])
df_sig['numero_oficial']        = normalizar_numerico(df_sig['numero_oficial'], 'numero_oficial', 'SIGSA')
df_sig['matricula_ruca']        = normalizar_numerico(df_sig['matricula_ruca'], 'matricula_ruca', 'SIGSA')
df_sig['cantidad_amparada']     = normalizar_numerico(df_sig['cantidad_amparada'], 'cantidad_amparada', 'SIGSA')
df_sig['cantidad_cierre']       = normalizar_numerico(df_sig['cantidad_cierre'], 'cantidad_cierre', 'SIGSA')
df_sig['dte_abierto']           = df_sig['estado'].astype(str).str.upper().str.strip().isin(['ABIERTO', 'SIN ARRIBO'])
df_sig['codigo_frigorifico']    = df_sig['codigo_frigorifico'].astype(str).str.strip()

import re

def extraer_num(s):
    if pd.isna(s): return pd.NA
    partes = re.split(r'[./]', str(s))
    nums = [p for p in partes if p.strip().isdigit() and len(p) > 2]
    return int(nums[-1]) if nums else pd.NA

rucas_siocal = set(df_pad['establecimiento_ruca'].dropna().astype(int))

# Mapeo Numero Oficial (entero) → RUCA desde SIGICA
MAP_NOFICIAL_INT_RUCA = (
    df_sigica[df_sigica['numero_oficial'].notna() & df_sigica['establecimiento_ruca'].notna()]
    .copy()
)
MAP_NOFICIAL_INT_RUCA['noficial_int'] = pd.to_numeric(
    MAP_NOFICIAL_INT_RUCA['numero_oficial'].astype(str).str.extract(r'(\d+)')[0],
    errors='coerce'
)
MAP_NOFICIAL_INT_RUCA = (
    MAP_NOFICIAL_INT_RUCA.dropna(subset=['noficial_int'])
    .drop_duplicates(subset='noficial_int')
    .set_index('noficial_int')['establecimiento_ruca']
    .to_dict()
)

# Mapeo Matricula RUCA → Establecimiento RUCA desde SIGICA
MAP_MATRICULA_RUCA = (
    df_sigica[df_sigica['matricula_ruca'].notna() & df_sigica['establecimiento_ruca'].notna()]
    .drop_duplicates(subset='matricula_ruca')
    .set_index('matricula_ruca')['establecimiento_ruca']
    .to_dict()
)

import re

def extraer_num(s):
    if pd.isna(s): return pd.NA
    partes = re.split(r'[./]', str(s))
    nums = [p for p in partes if p.strip().isdigit() and len(p) > 2]
    return int(nums[-1]) if nums else pd.NA

rucas_siocal = set(df_pad['establecimiento_ruca'].dropna().astype(int))

def resolver_ruca(row):
    num = extraer_num(row['codigo_frigorifico'])
    # Paso 1: número extraído está en SIOCAL → es el RUCA
    if pd.notna(num) and num in rucas_siocal:
        return num
    # Paso 2: tiene Numero Oficial → buscar en SIGICA por noficial
    if pd.notna(row['numero_oficial']):
        ruca = MAP_NOFICIAL_INT_RUCA.get(row['numero_oficial'], pd.NA)
        if pd.notna(ruca):
            return ruca
    # Paso 3: número extraído es Matricula RUCA → buscar en SIGICA por matricula
    if pd.notna(num):
        ruca = MAP_MATRICULA_RUCA.get(num, pd.NA)
        if pd.notna(ruca):
            return ruca
    return pd.NA

# Resolución de RUCA vectorizada (misma prioridad: SIOCAL -> nº oficial -> matrícula)
_num   = df_sig['codigo_frigorifico'].map(extraer_num)
_paso1 = _num.where(_num.isin(rucas_siocal))
_paso2 = df_sig['numero_oficial'].map(MAP_NOFICIAL_INT_RUCA)
_paso3 = _num.map(MAP_MATRICULA_RUCA)
_ruca  = _paso1.fillna(_paso2).fillna(_paso3)
df_sig['establecimiento_ruca'] = pd.array(pd.to_numeric(_ruca, errors='coerce'), dtype='Int64')

print(f'Resueltos: {df_sig["establecimiento_ruca"].notna().sum():,} de {len(df_sig):,}')
print(f'Sin resolver: {df_sig["establecimiento_ruca"].isna().sum():,}')

## 13. Capacidad Cámaras

In [ ]:
df_cam = leer_base(asignacion['capacidad_camaras'])
df_cam = fix_df_encoding(df_cam)
df_cam = renombrar_cols(df_cam, {
    'Numero Oficial':         'numero_oficial',
    'Razon Social':           'razon_social',
    'Identificacion Division':'id_division',
    'Tipo Division':          'tipo_division',
    'Capacidad Division':     'capacidad_division',
    'Establecimiento RUCA':   'establecimiento_ruca',
    'CUIT Titular':           'cuit_titular',
})
for campo in ['establecimiento_ruca', 'cuit_titular']:
    if campo not in df_cam.columns:
        df_cam[campo] = pd.NA
        print(f'⚠️  {campo} no encontrado en Capacidad Cámaras')
    elif campo == 'establecimiento_ruca':
        df_cam[campo] = normalizar_numerico(df_cam[campo], campo, 'Capacidad_Camaras')
    else:
        df_cam[campo] = normalizar_cuit(df_cam[campo])
df_cam['numero_oficial']     = normalizar_numerico(df_cam['numero_oficial'], 'numero_oficial', 'Capacidad_Camaras')
df_cam['capacidad_division'] = normalizar_numerico(df_cam['capacidad_division'], 'capacidad_division', 'Capacidad_Camaras')
df_cam['tipo_division']      = df_cam['tipo_division'].astype(str).str.lower().str.strip()
informe_base('CAPACIDAD CÁMARAS', df_cam)

# Resolver establecimiento_ruca via padrón SIGICA (mismo mapeo que SIGSA)
mask_sin_ruca = df_cam['establecimiento_ruca'].isna() & df_cam['numero_oficial'].notna()
if mask_sin_ruca.sum() > 0:
    df_cam.loc[mask_sin_ruca, 'establecimiento_ruca'] = (
        df_cam.loc[mask_sin_ruca, 'numero_oficial']
        .map(MAP_NOFICIAL_INT_RUCA)
    )
    resueltos = df_cam['establecimiento_ruca'].notna().sum()
    sin_resolver = df_cam['establecimiento_ruca'].isna().sum()
    print(f'Capacidad Cámaras: {resueltos:,} registros con RUCA resuelto | {sin_resolver:,} sin resolver')

In [ ]:
sin_resolver_cam = df_cam[df_cam['establecimiento_ruca'].isna() & df_cam['numero_oficial'].notna()]
print(f'Números oficiales sin resolver: {sin_resolver_cam["numero_oficial"].nunique()}')
print()
print('Muestra:')
print(sin_resolver_cam[['numero_oficial','razon_social','tipo_division']].drop_duplicates(subset='numero_oficial').head(10).to_string())

---
# Indicadores

Cada indicador se calcula en dos versiones donde corresponde:
- **Por planta** (`rol = Titular`): agrupado por `establecimiento_ruca`
- **Por usuario** (`rol = Usuario`): agrupado por `cuit_usuario_faena` + `establecimiento_ruca`

El usuario se deriva siempre desde `DTE_USUARIO` (tabla de mapeo DTe → usuario construida desde Ingresos).

In [ ]:
def ultimo_mes_con_datos(serie_fecha):
    fecha_max = serie_fecha.dropna().max()
    return fecha_max.year, fecha_max.month

def ultimo_periodo_30d(serie_fecha):
    """Últimos 30 días desde la fecha máxima disponible en la base."""
    fecha_max = serie_fecha.dropna().max().normalize()
    fecha_ini = fecha_max - pd.Timedelta(days=30)
    return fecha_ini, fecha_max

def mes_anterior(anio, mes):
    return (anio - 1, 12) if mes == 1 else (anio, mes - 1)

def filtrar_mes(df, col_fecha, anio, mes):
    return df[(df[col_fecha].dt.year == anio) & (df[col_fecha].dt.month == mes)]

# Razones sociales desde múltiples bases (cascada)
def extraer_razones(df, col_ruca='establecimiento_ruca', col_rs='razon_social'):
    return df[[col_ruca, col_rs]].dropna(subset=[col_rs]).drop_duplicates(subset=col_ruca)

rs_faena = extraer_razones(df_fae)
rs_aut   = extraer_razones(df_aut)
rs_tip   = extraer_razones(df_tip)
razones_sociales = (
    rs_faena
    .merge(rs_aut.rename(columns={'razon_social':'rs_aut'}), on='establecimiento_ruca', how='outer')
    .merge(rs_tip.rename(columns={'razon_social':'rs_tip'}), on='establecimiento_ruca', how='outer')
)
razones_sociales['razon_social'] = (
    razones_sociales['razon_social']
    .fillna(razones_sociales['rs_aut'])
    .fillna(razones_sociales['rs_tip'])
)
razones_sociales = razones_sociales[['establecimiento_ruca', 'razon_social']]

# CUITs en el padrón SIOCAL (para filtrar usuarios)
CUITS_SIOCAL = set(df_pad['cuit_titular'].dropna())

print(f'Razones sociales: {razones_sociales["razon_social"].notna().sum()} establecimientos con nombre')
print(f'CUITs en padrón SIOCAL: {len(CUITS_SIOCAL)}')


## Indicador 01 — % DTE Cerrados/Emitidos
**Base:** SIGSA. `cantidad_cierre / cantidad_amparada` por establecimiento.
**Por usuario:** cruce SIGSA × Ingresos por DTe para derivar el usuario de cada DTe.

In [ ]:
# ============ INDICADOR 01 — % DTE Cerrados sobre Emitidos ============
fecha_ini_01, fecha_fin_01 = ultimo_periodo_30d(df_sig['fecha_movimiento'])

df_sig_01 = df_sig[(df_sig['fecha_movimiento'] >= fecha_ini_01) & (df_sig['fecha_movimiento'] <= fecha_fin_01)]

ind01_planta = df_sig_01.groupby('establecimiento_ruca').agg(
    cantidad_amparada_total=('cantidad_amparada', 'sum'),
    cantidad_cierre_total=('cantidad_cierre', 'sum'),
).reset_index()
ind01_planta['01_pct_cerrado_amparado'] = (
    ind01_planta['cantidad_cierre_total'] / ind01_planta['cantidad_amparada_total'] * 100
).round(2)

print(f'Período: {fecha_ini_01.date()} → {fecha_fin_01.date()}')
print(f'Ind01 — Planta: {len(ind01_planta)}')

## Indicador 03 — % DTE con Faltante Ingresadas/Amparadas
**Bases:** SIGSA × Ingresos por DTe. Solo faltantes (sistema no permite sobrantes).
**Por usuario:** agrupado por `cuit_usuario_faena` + `establecimiento_ruca`.

In [ ]:
# ============ INDICADOR 03 — % DTE con Faltante Ingresadas/Amparadas ============
fecha_ini_03, fecha_fin_03 = ultimo_periodo_30d(df_ing['fecha_ingreso'])

sig_dte = df_sig[(df_sig['fecha_movimiento'] >= fecha_ini_03) & (df_sig['fecha_movimiento'] <= fecha_fin_03)].groupby('dte', as_index=False)['cantidad_amparada'].sum()
ing_dte = df_ing[(df_ing['fecha_ingreso'] >= fecha_ini_03) & (df_ing['fecha_ingreso'] <= fecha_fin_03)][['dte', 'cantidad_ingresada', 'establecimiento_ruca']].drop_duplicates(subset='dte')

cruce_ind03 = ing_dte.merge(sig_dte, on='dte', how='inner')
print(f'DTe cruzados: {len(cruce_ind03):,}')

cruce_ind03['diferencia'] = cruce_ind03['cantidad_ingresada'] - cruce_ind03['cantidad_amparada']
cruce_ind03['es_faltante'] = cruce_ind03['diferencia'] < 0
cruce_ind03['cabezas_faltante'] = np.where(cruce_ind03['es_faltante'], -cruce_ind03['diferencia'], 0)

ind03_planta = cruce_ind03.groupby('establecimiento_ruca').agg(
    total_dte=('dte', 'count'),
    dte_con_faltante=('es_faltante', 'sum'),
    cabezas_faltante_total=('cabezas_faltante', 'sum'),
).reset_index()
ind03_planta['03_pct_dte_con_faltante'] = (ind03_planta['dte_con_faltante'] / ind03_planta['total_dte'] * 100).round(2)

print(f'Período: {fecha_ini_03.date()} → {fecha_fin_03.date()}')
print(f'Ind03 — Planta: {len(ind03_planta)}')

## Indicador 04 — Concidencia de Sexo por DTE
Compara el sexo de los animales entre faena y movimientos de sigsa.

In [ ]:
# ============ INDICADOR 04 — Coincidencia de Sexo (SIGSA vs Faena) ============

MAP_FAENA_SEXO = {
    '>8D - NO': 'M', '>8D - TO': 'M', '>8D - VA': 'H',
    '2D - MEJ': 'M', '2D - NT':  'M', '2D - VQ':  'H',
    '4D - NT':  'M', '4D - TO':  'M', '4D - VQ':  'H',
    '6D - NO':  'M', '6D - TO':  'M', '6D - VA':  'H',
    '8D - NO':  'M', '8D - TO':  'M', '8D - VA':  'H',
}
MAP_SIGSA_SEXO = {
    'Vaca':       'H', 'Vaquillona': 'H', 'Ternera':    'H',
    'Novillo':    'M', 'Novillito':  'M', 'Toro':       'M',
    'Torito/MEJ': 'M', 'Ternero':    'M', 'Bueyes':     'M',
}

fecha_ini_04, fecha_fin_04 = ultimo_periodo_30d(df_fae['fecha_faena'])

sig_sexo = df_sig[df_sig['fecha_movimiento'] >= fecha_ini_04][['dte', 'categoria', 'cantidad_cierre', 'establecimiento_ruca']].copy()
sig_sexo['sexo_dte'] = sig_sexo['categoria'].map(MAP_SIGSA_SEXO)
sig_sexo = sig_sexo.dropna(subset=['sexo_dte'])

sig_por_sexo = (
    sig_sexo.groupby(['dte', 'sexo_dte'])['cantidad_cierre'].sum()
    .unstack(fill_value=0).reset_index()
)
sig_por_sexo = sig_por_sexo.rename(columns={'H': 'cabezas_H_sigsa', 'M': 'cabezas_M_sigsa'})
for col in ['cabezas_H_sigsa', 'cabezas_M_sigsa']:
    if col not in sig_por_sexo.columns: sig_por_sexo[col] = 0

fae_sexo = df_fae[df_fae['fecha_faena'] >= fecha_ini_04][['dte', 'categoria_faena', 'cantidad_cabezas', 'establecimiento_ruca']].copy()
fae_sexo['sexo_faena'] = fae_sexo['categoria_faena'].map(MAP_FAENA_SEXO)
fae_sexo = fae_sexo.dropna(subset=['sexo_faena'])

fae_por_sexo = (
    fae_sexo.groupby(['dte', 'sexo_faena'])['cantidad_cabezas'].sum()
    .unstack(fill_value=0).reset_index()
)
fae_por_sexo = fae_por_sexo.rename(columns={'H': 'cabezas_H_faena', 'M': 'cabezas_M_faena'})
for col in ['cabezas_H_faena', 'cabezas_M_faena']:
    if col not in fae_por_sexo.columns: fae_por_sexo[col] = 0

ruca_por_dte = fae_sexo[['dte', 'establecimiento_ruca']].drop_duplicates(subset='dte')

cruce_ind04 = sig_por_sexo.merge(fae_por_sexo, on='dte', how='inner')
cruce_ind04 = cruce_ind04.merge(ruca_por_dte, on='dte', how='left')

cruce_ind04['dif_H'] = (cruce_ind04['cabezas_H_sigsa'] - cruce_ind04['cabezas_H_faena']).abs()
cruce_ind04['dif_M'] = (cruce_ind04['cabezas_M_sigsa'] - cruce_ind04['cabezas_M_faena']).abs()
cruce_ind04['cabezas_no_coinciden'] = cruce_ind04['dif_H'] + cruce_ind04['dif_M']
cruce_ind04['total_cabezas_dte'] = cruce_ind04[['cabezas_H_faena', 'cabezas_M_faena']].sum(axis=1)

ind04_planta = cruce_ind04.groupby('establecimiento_ruca').agg(
    total_cabezas_cruzadas=('total_cabezas_dte', 'sum'),
    cabezas_no_coinciden=('cabezas_no_coinciden', 'sum'),
).reset_index()
ind04_planta['04_pct_no_coinciden_sexo'] = (
    ind04_planta['cabezas_no_coinciden'] / ind04_planta['total_cabezas_cruzadas'] * 100
).round(2)

print(f'Período: {fecha_ini_04.date()} → {fecha_fin_04.date()}')
print(f'Ind04 — Planta: {len(ind04_planta)}')


## Indicador 05 — Exceso Capacidad Corral
**Solo por planta.** Balance de stock diario en corrales con ventana de calentamiento de 20 días.

In [ ]:
# ============ INDICADOR 05 — Exceso Capacidad Corral ============

CAPACIDAD_CORRALES = (
    df_cam[df_cam['tipo_division'] == 'corral']
    .groupby('establecimiento_ruca')['capacidad_division'].sum()
    .rename('capacidad_total_corrales')
)

fecha_ini_05, fecha_fin_05 = ultimo_periodo_30d(df_ing['fecha_ingreso'])

ingresos_dia = (
    df_ing[(df_ing['fecha_ingreso'] >= fecha_ini_05) & (df_ing['fecha_ingreso'] <= fecha_fin_05)]
    .groupby(['establecimiento_ruca', df_ing['fecha_ingreso'].dt.normalize()])['cantidad_ingresada']
    .sum().rename('ingresos_dia').reset_index().rename(columns={'fecha_ingreso': 'fecha'})
)
ingresos_dia = ingresos_dia.merge(CAPACIDAD_CORRALES, on='establecimiento_ruca', how='left')
ingresos_dia = ingresos_dia.merge(razones_sociales, on='establecimiento_ruca', how='left')
ingresos_dia = ingresos_dia.merge(
    df_pad[['establecimiento_ruca','cuit_titular']].drop_duplicates(),
    on='establecimiento_ruca', how='left'
)
ingresos_dia['excede'] = ingresos_dia['ingresos_dia'] > ingresos_dia['capacidad_total_corrales']

ind05_maestra = ingresos_dia.groupby('establecimiento_ruca').agg(
    dias_con_exceso=('excede', 'sum'),
).reset_index()
ind05_maestra['05_pct_dias_exceso_corral'] = (
    ind05_maestra['dias_con_exceso'] / 30 * 100
).round(2)
ind05_planta = ind05_maestra[['establecimiento_ruca','05_pct_dias_exceso_corral']]

dias_con_alarma05 = ingresos_dia[ingresos_dia['excede']].copy()
dias_con_alarma05 = dias_con_alarma05.rename(columns={
    'ingresos_dia': 'cabezas_ingresadas',
    'capacidad_total_corrales': 'capacidad_corral',
})
dias_con_alarma05['alarma'] = True

sin_alarma05 = ingresos_dia[~ingresos_dia['establecimiento_ruca'].isin(dias_con_alarma05['establecimiento_ruca'])].copy()
sin_alarma05 = sin_alarma05.groupby(['establecimiento_ruca','razon_social','cuit_titular','capacidad_total_corrales']).agg(
    cabezas_ingresadas=('ingresos_dia','max')
).reset_index().rename(columns={'capacidad_total_corrales':'capacidad_corral'})
sin_alarma05['fecha'] = pd.NaT
sin_alarma05['alarma'] = False

cols_det05 = ['establecimiento_ruca','razon_social','cuit_titular','fecha','capacidad_corral','cabezas_ingresadas','alarma']
ind05_detalle = pd.concat([
    dias_con_alarma05[cols_det05],
    sin_alarma05[cols_det05],
], ignore_index=True).sort_values(['establecimiento_ruca','fecha'])

print(f'Período: {fecha_ini_05.date()} → {fecha_fin_05.date()}')
print(f'Ind05 — Planta: {len(ind05_planta)} ({(ind05_planta["05_pct_dias_exceso_corral"] > 0).sum()} con alerta)')


## Indicador 06 — #Emergencia/#Autorizaciones (mes actual vs anterior)
**Por planta:** directo desde Autorizaciones.
**Por usuario:** Autorizaciones × Ingresos por DTe.

In [ ]:
# ============ INDICADOR 06 — % Autorizaciones de Emergencia ============
fecha_ini_06, fecha_fin_06 = ultimo_periodo_30d(df_aut['fecha_faena'])
df_aut_06 = df_aut[(df_aut['fecha_faena'] >= fecha_ini_06) & (df_aut['fecha_faena'] <= fecha_fin_06)]

ind06_planta = df_aut_06.groupby('establecimiento_ruca').agg(
    total=('dte','count'),
    emergencias=('es_emergencia','sum'),
).reset_index()
ind06_planta['06_pct_emergencia_mes_actual'] = (ind06_planta['emergencias'] / ind06_planta['total'] * 100).round(2)

print(f'Período: {fecha_ini_06.date()} → {fecha_fin_06.date()}')
print(f'Ind06 — Planta: {len(ind06_planta)}')


## Indicador 07 — % Autorizaciones Posdatadas
**Por planta y por usuario** (mismo patrón que ind06).

In [ ]:
# ============ INDICADOR 07 — % Autorizaciones Posdatadas ============
fecha_ini_07, fecha_fin_07 = ultimo_periodo_30d(df_aut['fecha_faena'])
df_aut_07 = df_aut[(df_aut['fecha_faena'] >= fecha_ini_07) & (df_aut['fecha_faena'] <= fecha_fin_07)]

ind07_planta = df_aut_07.groupby('establecimiento_ruca').agg(
    total=('dte','count'),
    posdatadas=('es_posdatada','sum'),
).reset_index()
ind07_planta['07_pct_posdatadas'] = (ind07_planta['posdatadas'] / ind07_planta['total'] * 100).round(2)

print(f'Período: {fecha_ini_07.date()} → {fecha_fin_07.date()}')
print(f'Ind07 — Planta: {len(ind07_planta)}')


## Indicador 08 — % Aprobaciones Tardías (>10am)
**Por planta y por usuario.** Requiere `autorizacion_hora` en Autorizaciones.

In [ ]:
# ============ INDICADOR 08 — % Aprobaciones Tardías ============
# Tardía = aprobada el mismo día de la faena después de las 10am,
#          O aprobada el día posterior a la fecha de faena
fecha_ini_08, fecha_fin_08 = ultimo_periodo_30d(df_aut['fecha_faena'])
df_aut_08 = df_aut[
    (df_aut['fecha_faena'] >= fecha_ini_08) &
    (df_aut['fecha_faena'] <= fecha_fin_08) &
    df_aut['autorizacion_fecha_aprobada'].notna()
].copy()

if len(df_aut_08) > 0:
    df_aut_08['fecha_faena_date']      = df_aut_08['fecha_faena'].dt.normalize()
    df_aut_08['fecha_aprobada_date']   = df_aut_08['autorizacion_fecha_aprobada'].dt.normalize()
    df_aut_08['hora_aprobada']         = df_aut_08['autorizacion_fecha_aprobada'].dt.time
    limite_10am                        = pd.Timestamp(HORA_LIMITE_APROBACION).time()

    df_aut_08['aprobacion_tardia'] = (
        # Mismo día, después de las 10am
        ((df_aut_08['fecha_aprobada_date'] == df_aut_08['fecha_faena_date']) &
         (df_aut_08['hora_aprobada'] > limite_10am))
        |
        # Día posterior a la fecha de faena
        (df_aut_08['fecha_aprobada_date'] > df_aut_08['fecha_faena_date'])
    )

    ind08_planta = df_aut_08.groupby('establecimiento_ruca').agg(
        total_autorizaciones=('dte', 'count'),
        autorizaciones_tardias=('aprobacion_tardia', 'sum'),
    ).reset_index()
    ind08_planta['08_pct_aprobacion_tardia'] = (
        ind08_planta['autorizaciones_tardias'] / ind08_planta['total_autorizaciones'] * 100
    ).round(2)
else:
    print('⚠️  autorizacion_fecha_aprobada no disponible — indicador 08 vacío')
    ind08_planta = pd.DataFrame(columns=['establecimiento_ruca','total_autorizaciones','autorizaciones_tardias','08_pct_aprobacion_tardia'])

print(f'Período: {fecha_ini_08.date()} → {fecha_fin_08.date()}')
print(f'Ind08 — Planta: {len(ind08_planta)}')

## Indicador 09 — # Casos Faena con Demora (>3 días)
**Por planta y por usuario.** Cruce Ingresos × Faena por DTe.

In [ ]:
# ============ INDICADOR 09 — % DTe con Demora de Faena (>3 días) ============
fecha_ini_09, fecha_fin_09 = ultimo_periodo_30d(df_ing['fecha_ingreso'])

ing_dte9 = df_ing[
    (df_ing['fecha_ingreso'] >= fecha_ini_09) & (df_ing['fecha_ingreso'] <= fecha_fin_09)
][['dte','fecha_ingreso','establecimiento_ruca']].drop_duplicates(subset='dte')

fae_dte9 = df_fae.groupby('dte', as_index=False)['fecha_faena'].max().rename(columns={'fecha_faena':'fecha_faena_final'})
cruce_ind09 = ing_dte9.merge(fae_dte9, on='dte', how='inner')
cruce_ind09['dias_demora'] = (cruce_ind09['fecha_faena_final'] - cruce_ind09['fecha_ingreso']).dt.days
casos_demora = cruce_ind09[cruce_ind09['dias_demora'] > DEMORA_FAENA_DIAS]

ind09_aux = casos_demora.groupby('establecimiento_ruca').agg(
    casos_demora_mayor_3d=('dte','count'),
    promedio_dias_demora=('dias_demora','mean'),
).reset_index()
ind09_aux['promedio_dias_demora'] = ind09_aux['promedio_dias_demora'].round(1)
total09 = cruce_ind09.groupby('establecimiento_ruca')['dte'].count().rename('total_dte_cruzados').reset_index()
ind09_planta = ind09_aux.merge(total09, on='establecimiento_ruca', how='left')
ind09_planta['09_pct_dte_con_demora'] = (
    ind09_planta['casos_demora_mayor_3d'] / ind09_planta['total_dte_cruzados'] * 100
).round(2)

print(f'Período: {fecha_ini_09.date()} → {fecha_fin_09.date()}')
print(f'Ind09 — Planta: {len(ind09_planta)}')


## Indicador 10 — % Tropas con Dentición Uniforme
**Solo por planta.** Requiere `numero_tropa` en Tipificación.

In [ ]:
if TIENE_NUMERO_TROPA:
    # MIN_ANIMALES_TROPA ahora viene de la celda "0.1 · Parámetros desde Excel"
    tropas10 = df_tip.groupby(['establecimiento_ruca','numero_tropa']).agg(
        cantidad_animales=('cantidad_tipificacion','sum'),
        denticiones_distintas=('denticion','nunique'),
    ).reset_index()
    tropas10 = tropas10[tropas10['cantidad_animales'] > MIN_ANIMALES_TROPA]
    tropas10['tropa_homogenea'] = tropas10['denticiones_distintas'] == 1
    ind10_planta = tropas10.groupby('establecimiento_ruca').agg(
        total_tropas_evaluadas=('tropa_homogenea','count'),
        tropas_homogeneas=('tropa_homogenea','sum'),
    ).reset_index()
    ind10_planta['10_pct_tropas_denticion_uniforme'] = (
        ind10_planta['tropas_homogeneas'] / ind10_planta['total_tropas_evaluadas'] * 100
    ).round(2)
    print(f'Ind10 — Planta: {len(ind10_planta)}')
else:
    ind10_planta = pd.DataFrame(columns=['establecimiento_ruca','10_pct_tropas_denticion_uniforme'])
    print('⚠️  Ind10 deshabilitado — falta numero_tropa en Tipificación')


## Indicador 11 — Tipificador No Habilitado en Planta
**Solo por planta.**

In [ ]:
plantas_habilitadas_por_tipificador = (
    df_hab[df_hab['habilitado']]
    .groupby('nro_tipificador')['establecimiento_ruca']
    .apply(set)
)
def tipificador_ok(row):
    plantas = plantas_habilitadas_por_tipificador.get(row['numero_tipificador'])
    if plantas is None: return False
    return row['establecimiento_ruca'] in plantas

ultima_fecha_por_ruca = (
    df_tip.groupby('establecimiento_ruca')['fecha_faena']
    .max().dt.normalize().rename('ultima_fecha_tip').reset_index()
)
df_tip_valid = (
    df_tip.dropna(subset=['numero_tipificador','establecimiento_ruca'])
    .merge(ultima_fecha_por_ruca, on='establecimiento_ruca', how='left')
)
df_tip_valid = df_tip_valid[
    df_tip_valid['fecha_faena'].dt.normalize() == df_tip_valid['ultima_fecha_tip']
].copy()
df_tip_valid['tipificador_ok'] = df_tip_valid.apply(tipificador_ok, axis=1)

ind11_planta = df_tip_valid.groupby('establecimiento_ruca').agg(
    total_registros=('tipificador_ok','count'),
    registros_no_habilitados=('tipificador_ok', lambda x: (~x).sum()),
).reset_index()
ind11_planta['11_tipificador_no_habilitado'] = ind11_planta['registros_no_habilitados'] > 0
print(f'Ind11 — Planta: {len(ind11_planta)} ({ind11_planta["11_tipificador_no_habilitado"].sum()} con alerta)')


## Indicador 12 — Entropía de Tipificación
**Solo por planta.** Entropía de Shannon por tipificador × establecimiento.
Score = (1 - entropía) × confianza muestral.

In [ ]:
CATEGORIAS_TIP = ['A0','A1','A2','A3','A4','B0','B1','B2','B3','B4',
                  'C0','C1','C2','C3','C4','D0','D1','D2','D3','D4',
                  'E0','E1','E2','E3','E4']
N_CATEGORIAS = len(CATEGORIAS_TIP)

pivot = (
    df_tip
    .groupby(['numero_tipificador','establecimiento_ruca','tipo_tipificacion'])['cantidad_tipificacion']
    .sum().reset_index()
    .pivot_table(index=['numero_tipificador','establecimiento_ruca'],
                 columns='tipo_tipificacion', values='cantidad_tipificacion',
                 aggfunc='sum', fill_value=0)
    .reset_index()
)
for cat in CATEGORIAS_TIP:
    if cat not in pivot.columns: pivot[cat] = 0

conteos = pivot[CATEGORIAS_TIP].values.astype(float)
totales = conteos.sum(axis=1)
with np.errstate(divide='ignore', invalid='ignore'):
    p = np.where(conteos > 0, conteos / totales[:, None], 0)
    entropia_raw = -np.sum(np.where(p > 0, p * np.log(p), 0), axis=1)
    pivot['entropia']          = np.where(totales > 0, entropia_raw / np.log(N_CATEGORIAS), 0).round(4)
pivot['confianza_muestral'] = np.minimum(totales / 100, 1).round(4)
pivot['score_alerta']       = ((1 - pivot['entropia']) * pivot['confianza_muestral']).round(4)
pivot['total_animales']     = totales.astype(int)

def semaforo(row):
    if row['total_animales'] < 30:   return 'SIN DATOS'
    if row['score_alerta'] >= 0.7:   return 'ALERTA ALTA'
    if row['score_alerta'] >= 0.4:   return 'ALERTA MEDIA'
    if row['score_alerta'] >= 0.15:  return 'REVISAR'
    return 'NORMAL'
pivot['semaforo'] = pivot.apply(semaforo, axis=1)

ind12_detalle = pivot.merge(razones_sociales, on='establecimiento_ruca', how='left')
cols_orden = ['establecimiento_ruca','razon_social','numero_tipificador','total_animales'] + \
             CATEGORIAS_TIP + ['entropia','confianza_muestral','score_alerta','semaforo']
ind12_detalle = ind12_detalle[cols_orden].sort_values(['establecimiento_ruca','score_alerta'], ascending=[True,False])

ind12_planta = (
    ind12_detalle[ind12_detalle['semaforo'] == 'ALERTA ALTA']
    .groupby('establecimiento_ruca').size()
    .reset_index(name='12_tipificadores_alerta_alta')
)
# Para establecimientos con datos pero sin ALERTA ALTA, poner 0
rucas_con_tip = set(df_tip['establecimiento_ruca'].dropna())
ind12_planta_full = pd.DataFrame({'establecimiento_ruca': list(rucas_con_tip)})
ind12_planta_full = ind12_planta_full.merge(ind12_planta, on='establecimiento_ruca', how='left')
ind12_planta_full['12_tipificadores_alerta_alta'] = ind12_planta_full['12_tipificadores_alerta_alta'].fillna(0).astype(int)
ind12_planta = ind12_planta_full

print(f'Ind12 — Detalle: {len(ind12_detalle)} combinaciones tipificador-planta')
print(f'       Planta: {len(ind12_planta)} ({(ind12_planta["12_tipificadores_alerta_alta"] > 0).sum()} con al menos 1 ALERTA ALTA)')

# --- Cabezas bovinas tipificadas por planta, último mes (para % faena tipificada) ---
_ult_mes_tip = int(df_tip['numero_mes'].max())
tip_ultimo_mes = (
    df_tip[(df_tip['numero_mes'] == _ult_mes_tip) &
           (df_tip['especie'] == 'Bovinos') &
           (df_tip['eliminada'] == 0)]
    .groupby('establecimiento_ruca')['cantidad_tipificacion'].sum()
    .reset_index(name='cabezas_tipificadas_mes')
)
print('Último mes tipif:', _ult_mes_tip, '| plantas con tipificación bovina:', len(tip_ultimo_mes))


## Indicador 13 — Faena Mayor a Ingreso
**Por planta y por usuario.**

In [ ]:
# ============ INDICADOR 13 — % Faenado sobre Ingresado (últimos 30 días) ============
fecha_ini_13, fecha_fin_13 = ultimo_periodo_30d(df_ing['fecha_ingreso'])

ing_mes = df_ing[
    (df_ing['fecha_ingreso'] >= fecha_ini_13) & (df_ing['fecha_ingreso'] <= fecha_fin_13)
][['dte', 'cantidad_ingresada', 'establecimiento_ruca']].drop_duplicates(subset='dte')

fae_cab_dte = df_fae.groupby('dte', as_index=False)['cantidad_cabezas'].sum().rename(
    columns={'cantidad_cabezas': 'cabezas_faenadas'}
)
cruce_ind13 = ing_mes.merge(fae_cab_dte, on='dte', how='left')
cruce_ind13['cabezas_faenadas'] = cruce_ind13['cabezas_faenadas'].fillna(0)

ind13_planta = cruce_ind13.groupby('establecimiento_ruca').agg(
    total_dte=('dte', 'count'),
    cabezas_ingresadas_total=('cantidad_ingresada', 'sum'),
    cabezas_faenadas_total=('cabezas_faenadas', 'sum'),
).reset_index()
ind13_planta['13_pct_faenado_sobre_ingresado'] = (
    ind13_planta['cabezas_faenadas_total'] / ind13_planta['cabezas_ingresadas_total'] * 100
).round(2)

print(f'Período: {fecha_ini_13.date()} → {fecha_fin_13.date()}')
print(f'Ind13 — Planta: {len(ind13_planta)}')


## 14. Indicador 14 - Cupo de Faena

In [ ]:
# ============ HABILITACIONES DE FAENA — Tránsito + Indicador 14 ============

df_hab14 = leer_base(asignacion['habilitacion_faena'], header=None, skiprows=4)
df_hab14.columns = [
    'cuit','razon_social','establecimiento_ruca','provincia','localidad','tipo_hab','categoria_hab','cupo_faena',
    'bov_cupo_diario','bov_faena_dia_orig','bov_excedido_orig',
    'porc_cupo_diario','porc_faena_dia_orig',
    'ovc_cupo_diario','ovc_faena_dia_orig',
    'bov_cupo_mensual','bov_faena_mes_orig','bov_excedido2_orig',
    'porc_cupo_mensual','porc_faena_mes_orig',
    'ovc_cupo_mensual','ovc_faena_mes_orig',
]
df_hab14['establecimiento_ruca'] = pd.to_numeric(df_hab14['establecimiento_ruca'], errors='coerce')
df_hab14 = df_hab14.dropna(subset=['establecimiento_ruca'])
df_hab14['establecimiento_ruca'] = df_hab14['establecimiento_ruca'].astype(int)
df_hab14['tipo_hab']     = df_hab14['tipo_hab'].astype(str).str.strip().str.upper()
df_hab14['categoria_hab']= df_hab14['categoria_hab'].astype(str).str.strip()

# --- Columna Tránsito ---
df_hab14['transito'] = df_hab14['tipo_hab'].map({
    'NACIONAL':   'Federal',
    'PROVINCIAL': 'Provincial',
})

# --- Indicador 14 — Romaneo vs Cupo ---
fecha_fin_14 = df_fae['fecha_faena'].dropna().max().normalize()
fecha_ini_14 = fecha_fin_14.replace(day=1)

# Mapeo especie Faena → grupo
def mapear_especie(especie):
    e = str(especie).strip().lower()
    if 'bovino' in e: return 'bov'
    if 'porcino' in e: return 'porc'
    if 'ovino' in e or 'caprino' in e: return 'ovc'
    return None

df_fae_14 = df_fae[(df_fae['fecha_faena'] >= fecha_ini_14) & (df_fae['fecha_faena'] <= fecha_fin_14)].copy()
df_fae_14['grupo_especie'] = df_fae_14['especie'].apply(mapear_especie)

resultados_14 = []

for prefijo, nombre_especie in [('bov', 'Bovino'), ('porc', 'Porcino'), ('ovc', 'Ovino/Caprino')]:
    col_diario  = f'{prefijo}_cupo_diario'
    col_mensual = f'{prefijo}_cupo_mensual'

    fae_esp = df_fae_14[df_fae_14['grupo_especie'] == prefijo]

    # Faena por establecimiento: suma total y días con faena
    fae_resumen = fae_esp.groupby('establecimiento_ruca').agg(
        cabezas_total=('cantidad_cabezas', 'sum'),
        dias_con_faena=('fecha_faena', lambda x: x.dt.normalize().nunique()),
    ).reset_index()

    for _, row_hab in df_hab14.iterrows():
        ruca        = int(row_hab['establecimiento_ruca'])
        cupo_diario = row_hab[col_diario]
        cupo_mensual= row_hab[col_mensual]

        fila_fae = fae_resumen[fae_resumen['establecimiento_ruca'] == ruca]
        if len(fila_fae) == 0:
            continue

        cabezas_total  = fila_fae['cabezas_total'].iloc[0]
        dias_con_faena = fila_fae['dias_con_faena'].iloc[0]

        if pd.notna(cupo_diario) and dias_con_faena > 0:
            faena_calc = cabezas_total / dias_con_faena
            cupo       = cupo_diario
            tipo_cupo  = 'Diario'
        elif pd.notna(cupo_mensual):
            faena_calc = cabezas_total
            cupo       = cupo_mensual
            tipo_cupo  = 'Mensual'
        else:
            continue

        resultados_14.append({
            'establecimiento_ruca': ruca,
            'especie':              nombre_especie,
            'tipo_cupo':            tipo_cupo,
            'cupo':                 cupo,
            'faena_calculada':      round(faena_calc, 1),
            'diferencia':           round(faena_calc - cupo, 1),
            'excedido':             faena_calc > cupo,
        })

ind14_detalle = pd.DataFrame(resultados_14)
ind14_detalle = ind14_detalle.merge(razones_sociales, on='establecimiento_ruca', how='left')
ind14_detalle = ind14_detalle.merge(
    df_pad[['establecimiento_ruca','cuit_titular']].drop_duplicates(),
    on='establecimiento_ruca', how='left'
)
ind14_detalle = ind14_detalle[[
    'establecimiento_ruca','cuit_titular','razon_social',
    'especie','tipo_cupo','cupo','faena_calculada','diferencia','excedido'
]]

# Tabla maestra: binaria por establecimiento (excedido en al menos 1 especie)
ind14_planta = (
    ind14_detalle.groupby('establecimiento_ruca')['excedido']
    .any().reset_index().rename(columns={'excedido': '14_romaneo_excede_cupo'})
)

# Columna tránsito para la tabla maestra
transito_map = df_hab14[['establecimiento_ruca','transito']].drop_duplicates(subset='establecimiento_ruca')

print(f'Período: {fecha_ini_14.date()} → {fecha_fin_14.date()}')
print(f'Ind14 — Detalle: {len(ind14_detalle)} filas | Planta: {len(ind14_planta)} ({ind14_planta["14_romaneo_excede_cupo"].sum()} con alerta)')
print(f'Tránsito — {len(transito_map)} establecimientos mapeados')
print(df_hab14['transito'].value_counts())

## Indicador 15 — Stock > Capacidad Cámara
**Solo por planta.**

In [ ]:
# ============ INDICADOR 15 — Stock > Capacidad Cámara ============

CAPACIDAD_CAMARAS = (
    df_cam[df_cam['tipo_division'] == 'camara']
    .groupby('establecimiento_ruca')['capacidad_division'].sum()
    .rename('capacidad_camara_total')
)

fecha_ini_15, fecha_fin_15 = ultimo_periodo_30d(df_exi['fecha_existencia'])

exi_periodo = df_exi[
    (df_exi['fecha_existencia'] >= fecha_ini_15) &
    (df_exi['fecha_existencia'] <= fecha_fin_15)
].copy()

exi_con_cap = exi_periodo.merge(CAPACIDAD_CAMARAS, on='establecimiento_ruca', how='left')
exi_con_cap = exi_con_cap.merge(razones_sociales, on='establecimiento_ruca', how='left')

# Convertir todo a medias reses usando columna 'corte'
MAP_CORTE_FACTOR = {
    '1/2 Res':        1.0,
    '1/2 Reses':      1.0,
    '1/4 Traseros':   0.5,
    '1/4 Delanteros': 0.5,
}
exi_con_cap['factor_corte'] = exi_con_cap['corte'].map(MAP_CORTE_FACTOR).fillna(1.0)
exi_con_cap['cantidad_medias_reses'] = exi_con_cap['cantidad'] * exi_con_cap['factor_corte']

# Agrupar por establecimiento y fecha para comparar contra capacidad
exi_agrupado = exi_con_cap.groupby(['establecimiento_ruca','razon_social','fecha_existencia','capacidad_camara_total']).agg(
    cantidad_medias_reses=('cantidad_medias_reses','sum')
).reset_index()
exi_agrupado['excede'] = exi_agrupado['cantidad_medias_reses'] > exi_agrupado['capacidad_camara_total']

# Tabla maestra
ind15_maestra = exi_agrupado.groupby('establecimiento_ruca').agg(
    total_registros=('cantidad_medias_reses','count'),
    dias_con_sobrestock=('excede','sum'),
).reset_index()
ind15_maestra['15_pct_dias_sobrestock'] = (
    ind15_maestra['dias_con_sobrestock'] / ind15_maestra['total_registros'] * 100
).round(2)
ind15_planta = ind15_maestra[['establecimiento_ruca','15_pct_dias_sobrestock']]

# Desglosado
dias_con_alarma15 = exi_agrupado[exi_agrupado['excede']].copy()
dias_con_alarma15 = dias_con_alarma15.rename(columns={
    'fecha_existencia': 'fecha',
    'cantidad_medias_reses': 'stock_medias_reses',
    'excede': 'alarma',
})

sin_alarma15 = exi_agrupado[~exi_agrupado['establecimiento_ruca'].isin(dias_con_alarma15['establecimiento_ruca'])].copy()
sin_alarma15 = sin_alarma15.groupby(['establecimiento_ruca','razon_social','capacidad_camara_total']).agg(
    stock_medias_reses=('cantidad_medias_reses','max')
).reset_index()
sin_alarma15['fecha'] = pd.NaT
sin_alarma15['alarma'] = False

cols_det15 = ['establecimiento_ruca','razon_social','fecha','capacidad_camara_total','stock_medias_reses','alarma']
ind15_detalle = pd.concat([
    dias_con_alarma15[[c for c in cols_det15 if c in dias_con_alarma15.columns]],
    sin_alarma15[[c for c in cols_det15 if c in sin_alarma15.columns]],
], ignore_index=True).sort_values(['establecimiento_ruca','fecha'])

print(f'Período: {fecha_ini_15.date()} → {fecha_fin_15.date()}')
print(f'Ind15 — Planta: {len(ind15_planta)} ({(ind15_planta["15_pct_dias_sobrestock"] > 0).sum()} con alerta)')

## Indicador 16 — Faena sin Actualización de Existencias
**Solo por planta.** Compara días con registro en Faena vs días con registro en Existencias.

In [ ]:
# ============ INDICADOR 16 — Faena sin Actualización de Existencias ============
fecha_ini_16, fecha_fin_16 = ultimo_periodo_30d(df_fae['fecha_faena'])
fecha_ini_16 = max(fecha_ini_16, df_exi['fecha_existencia'].dropna().min().normalize())

dias_faena = (
    df_fae[(df_fae['fecha_faena'] >= fecha_ini_16) & (df_fae['fecha_faena'] <= fecha_fin_16)]
    .groupby('establecimiento_ruca')['fecha_faena']
    .nunique().rename('dias_con_faena').reset_index()
)
dias_exi = (
    df_exi[(df_exi['fecha_existencia'] >= fecha_ini_16) & (df_exi['fecha_existencia'] <= fecha_fin_16)]
    .groupby('establecimiento_ruca')['fecha_existencia']
    .nunique().rename('dias_con_existencias').reset_index()
)
ind16_planta = dias_faena.merge(dias_exi, on='establecimiento_ruca', how='outer').fillna(0)
ind16_planta['dias_con_faena']       = ind16_planta['dias_con_faena'].astype(int)
ind16_planta['dias_con_existencias'] = ind16_planta['dias_con_existencias'].astype(int)
ind16_planta['dias_sin_declarar']    = ind16_planta['dias_con_faena'] - ind16_planta['dias_con_existencias']

ind16_planta['16_pct_faena_sin_existencias'] = pd.to_numeric(
    ind16_planta['dias_sin_declarar'].clip(lower=0) / ind16_planta['dias_con_faena'].replace(0, pd.NA) * 100,
    errors='coerce'
).round(2)

ind16_planta = ind16_planta.merge(razones_sociales, on='establecimiento_ruca', how='left')
print(f'Período: {fecha_ini_16.date()} → {fecha_fin_16.date()}')
print(f'Ind16 — Planta: {len(ind16_planta)}')


---
# Tabla Maestra

Una fila por combinación CUIT × establecimiento_ruca × rol.
- **Titular:** dueño de la planta (viene del padrón SIOCAL)
- **Usuario:** opera en una planta ajena (solo los 216 CUITs del padrón que aparecen en Ingresos como usuarios)

Columnas base: `establecimiento_ruca`, `razon_social`, `cuit`, `provincia`, `actividad`, `especies`, `rol`, `total_cabezas_ingresadas`

Indicadores solo por planta: 05, 10, 11, 12, 15, 16
Indicadores por planta y usuario: 01, 03, 06, 07, 08, 09, 13

In [ ]:
# === TAMAÑO Y REGIÓN ===

# Tamaño: basado en promedio histórico de cabezas faenadas por día
faena_por_mes_hist = (
    df_fae.groupby(['establecimiento_ruca', df_fae['fecha_faena'].dt.to_period('M')])
    .agg(
        cabezas_mes=('cantidad_cabezas', 'sum'),
        dias_con_faena=('fecha_faena', lambda x: x.dt.normalize().nunique())
    ).reset_index()
)
faena_por_mes_hist['cabezas_dia'] = faena_por_mes_hist['cabezas_mes'] / faena_por_mes_hist['dias_con_faena']
tamanio_hist = faena_por_mes_hist.groupby('establecimiento_ruca')['cabezas_dia'].mean().round(1).reset_index()

def clasificar_tamanio(cab_dia):
    if pd.isna(cab_dia): return pd.NA
    if cab_dia <= TAMANIO_CHICO_MAX:    return 'C - Chico'
    if cab_dia <= TAMANIO_MEDIANO_MAX:  return 'B - Mediano'
    return 'A - Grande'

tamanio_hist['tamanio'] = tamanio_hist['cabezas_dia'].apply(clasificar_tamanio)
tamanio_map = tamanio_hist[['establecimiento_ruca','tamanio']]

# Región
MAP_REGION = {
    'BUENOS AIRES': 'Pampeana', 'SANTA FE': 'Pampeana', 'CORDOBA': 'Pampeana',
    'ENTRE RIOS': 'Pampeana', 'LA PAMPA': 'Pampeana',
    'CORRIENTES': 'NEA', 'CHACO': 'NEA', 'FORMOSA': 'NEA', 'MISIONES': 'NEA',
    'SALTA': 'NOA', 'JUJUY': 'NOA', 'TUCUMAN': 'NOA',
    'SANTIAGO DEL ESTERO': 'NOA', 'CATAMARCA': 'NOA', 'LA RIOJA': 'NOA',
    'MENDOZA': 'Cuyo', 'SAN JUAN': 'Cuyo', 'SAN LUIS': 'Cuyo',
    'NEUQUEN': 'Patagonia', 'RIO NEGRO': 'Patagonia', 'CHUBUT': 'Patagonia',
    'SANTA CRUZ': 'Patagonia', 'TIERRA DEL FUEGO': 'Patagonia',
}
region_map = df_pad[['establecimiento_ruca','provincia']].copy()
region_map['region'] = region_map['provincia'].map(MAP_REGION)

print(f'Tamaño — {tamanio_map["tamanio"].value_counts().to_dict()}')
print(f'Región — {region_map["region"].value_counts().to_dict()}')

In [ ]:
# --- TABLA MAESTRA: una fila por establecimiento ---
# Base madre = padrón SIOCAL

maestra = df_pad[[
    'establecimiento_ruca','cuit_titular','razon_social',
    'provincia','actividad','especies','estado'
]].copy()
maestra = maestra.rename(columns={'cuit_titular':'cuit'})

# Cabezas ingresadas (últimos 30 días)
fecha_ini_cab, fecha_fin_cab = ultimo_periodo_30d(df_ing['fecha_ingreso'])
cabezas_planta = (
    df_ing[(df_ing['fecha_ingreso'] >= fecha_ini_cab) & (df_ing['fecha_ingreso'] <= fecha_fin_cab)]
    .groupby('establecimiento_ruca')['cantidad_ingresada']
    .sum().rename('total_cabezas_ingresadas').reset_index()
)
maestra = maestra.merge(cabezas_planta, on='establecimiento_ruca', how='left')

# Scoring
maestra = maestra.merge(df_scoring[['establecimiento_ruca','ultima_inspeccion','scoring_actual']], on='establecimiento_ruca', how='left')
maestra['ultima_inspeccion'] = pd.to_datetime(maestra['ultima_inspeccion'], errors='coerce').dt.strftime('%d-%m-%Y')
maestra = maestra.merge(transito_map, on='establecimiento_ruca', how='left')
maestra = maestra.merge(tamanio_map, on='establecimiento_ruca', how='left')
maestra = maestra.merge(region_map[['establecimiento_ruca','region']], on='establecimiento_ruca', how='left')
# Función para pegar indicadores
def pegar_planta(df, df_ind, cols, on='establecimiento_ruca'):
    cols_presentes = [c for c in cols if c in df_ind.columns]
    return df.merge(df_ind[[on] + cols_presentes], on=on, how='left')

# Pegar indicadores en orden (04 después del 10)
maestra = pegar_planta(maestra, ind01_planta,  ['01_pct_cerrado_amparado'])
maestra = pegar_planta(maestra, ind03_planta,  ['03_pct_dte_con_faltante'])
maestra = pegar_planta(maestra, ind04_planta,  ['04_pct_no_coinciden_sexo'])
maestra = pegar_planta(maestra, ind05_planta,  ['05_pct_dias_exceso_corral'])
maestra = pegar_planta(maestra, ind06_planta,  ['06_pct_emergencia_mes_actual'])
maestra = pegar_planta(maestra, ind07_planta,  ['07_pct_posdatadas'])
maestra = pegar_planta(maestra, ind08_planta,  ['08_pct_aprobacion_tardia'])
maestra = pegar_planta(maestra, ind09_planta,  ['09_pct_dte_con_demora'])
maestra = pegar_planta(maestra, ind10_planta,  ['10_pct_tropas_denticion_uniforme'])
maestra = pegar_planta(maestra, ind11_planta,  ['11_tipificador_no_habilitado'])
maestra = pegar_planta(maestra, ind12_planta,  ['12_tipificadores_alerta_alta'])
maestra = pegar_planta(maestra, ind13_planta,  ['13_pct_faenado_sobre_ingresado'])
maestra = pegar_planta(maestra, ind14_planta,  ['14_romaneo_excede_cupo'])
maestra = pegar_planta(maestra, ind15_planta,  ['15_pct_dias_sobrestock'])
maestra = pegar_planta(maestra, ind16_planta,  ['16_pct_faena_sin_existencias'])

# Renombres finales
maestra = maestra.rename(columns={
    'establecimiento_ruca':              'Establecimiento SIOCAL',
    'razon_social':                      'Razón Social',
    'cuit':                              'CUIT',
    'provincia':                         'Provincia',
    'actividad':                         'Actividad',
    'especies':                          'Especies',
    'estado':                            'Estado',
    'total_cabezas_ingresadas':          'Total Cabezas Ingresadas',
    'ultima_inspeccion':                 'Última Inspección',
    'scoring_actual':                    'Scoring Actual',
    '01_pct_cerrado_amparado':           '01 - % DTE Cerrados sobre Emitidos',
    '03_pct_dte_con_faltante':           '03 - % DTE con Faltante Ingresadas/Amparadas',
    '05_pct_dias_exceso_corral':         '05 - % Días con Exceso Capacidad Corral',
    '06_pct_emergencia_mes_actual':      '06 - % Autorizaciones de Emergencia',
    '07_pct_posdatadas':                 '07 - % Autorizaciones Posdatadas',
    '08_pct_aprobacion_tardia':          '08 - % Aprobaciones Tardías (>10am)',
    '09_pct_dte_con_demora':             '09 - % DTe con Demora de Faena',
    '04_pct_no_coinciden_sexo':          '04 - % Cabezas con No Coincidencia de Sexo',
    '10_pct_tropas_denticion_uniforme':  '10 - % Tropas con Dentición Uniforme',
    '11_tipificador_no_habilitado':      '11 - Tipificador No Habilitado en Planta',
    '12_tipificadores_alerta_alta':      '12 - Tipificadores con Alerta Alta (Entropía)',
    '13_pct_faenado_sobre_ingresado':    '13 - % Faenado sobre Ingresado (últimos 30 días)',
    '15_pct_dias_sobrestock':            '15 - % Días con Stock > Capacidad Cámara',
    '16_pct_faena_sin_existencias':      '16 - % Días Faena sin Existencias',
    'transito':                          'Tránsito',
    '14_romaneo_excede_cupo':            '14 - Romaneo Excede Cupo de Faena',
    'tamanio': 'Tamaño',
'region':  'Región',
})

print(f'Tabla maestra: {len(maestra)} establecimientos')
print(f'Columnas: {len(maestra.columns)}')
maestra.head(3)


## Semáforo por Grupo

Agrupa los 15 indicadores en 5 grupos. Color del grupo = worst-of de los indicadores con dato (el riesgo siempre manda); **VIOLETA** = ningún indicador con dato = no declara nada en esa etapa. El **14** se excluye cuando viene gris (no aplica). `SCORE` = puntos de indicadores (rojo 5 / amarillo 1) + `PUNTOS_VIOLETA` por cada violeta, que suma **solo si la planta operó** (cabezas > 0). `grises · GX` = indicadores del grupo sin dato (para filtrar). Umbrales editables en `UMBRALES`.

In [ ]:
# --- Semáforo por Grupo (worst-of + violeta) ---
# GRUPOS, UMBRALES, EXCLUIR_SI_GRIS, PUNTOS y PUNTOS_VIOLETA ahora vienen de la celda
# "0.1 · Parámetros desde Excel" (config_parametros_tablero.xlsx).
IND_COLS = {c[:2]: c for c in maestra.columns if c[:2].isdigit()}
_ORDEN = {'ROJO': 3, 'AMARILLO': 2, 'VERDE': 1}

def _color(code, v):
    if pd.isna(v): return 'GRIS'
    tipo, amar, rojo = UMBRALES[code]
    if tipo == 'flag':   return 'ROJO' if v >= 1 else 'VERDE'
    if tipo == 'conteo': return 'ROJO' if v >= rojo else ('AMARILLO' if v >= amar else 'VERDE')
    if tipo == 'alto':   return 'ROJO' if v >= rojo else ('AMARILLO' if v >= amar else 'VERDE')
    if tipo == 'bajo':   return 'ROJO' if v <  rojo else ('AMARILLO' if v <  amar else 'VERDE')

def _color_grupo(codigos, colores):
    efectivos = [c for code, c in zip(codigos, colores) if not (c == 'GRIS' and code in EXCLUIR_SI_GRIS)]
    reales = [c for c in efectivos if c != 'GRIS']
    if reales:
        return max(reales, key=lambda c: _ORDEN[c])
    return 'GRIS'

# matriz de colores por indicador (una sola vez)
colores_ind = pd.DataFrame({
    code: pd.to_numeric(maestra[IND_COLS[code]], errors='coerce').apply(lambda v, c=code: _color(c, v))
    for code in UMBRALES
})

# columnas descriptivas = todo lo que NO es indicador
desc_cols = [c for c in maestra.columns if not c[:2].isdigit()]
maestra_grupos   = maestra[desc_cols].copy()
maestra_desglose = maestra[desc_cols].copy()
maestra_grupos = maestra_grupos.rename(columns={'Scoring Actual': 'Score Fiscalización'})

for nombre_grupo, codigos in GRUPOS.items():
    maestra_grupos[nombre_grupo] = colores_ind[codigos].apply(lambda r: _color_grupo(codigos, list(r)), axis=1)
    maestra_grupos[f'grises · {nombre_grupo[:2]}'] = (colores_ind[codigos] == 'GRIS').sum(axis=1)
    for code in codigos:
        maestra_desglose[IND_COLS[code]] = colores_ind[code]

# SCORE = puntos de indicadores + violetas (la violeta suma solo si la planta operó)
opero = pd.to_numeric(maestra['Total Cabezas Ingresadas'], errors='coerce').fillna(0) > 0
n_gris_total = sum((maestra_grupos[g] == 'GRIS').astype(int) for g in GRUPOS)
maestra_grupos['Score Semáforo'] = colores_ind.replace(PUNTOS).sum(axis=1) + n_gris_total * PUNTOS_VIOLETA * opero.astype(int)

# Score Fiscalización (mochila 1-4). Los que no tienen -> 2 imputado (se marca aparte)
fisc = pd.to_numeric(maestra_grupos['Score Fiscalización'], errors='coerce')
fisc = fisc.mask(fisc == 0)                            # el 0 = "sin scoring" -> NaN, se imputa igual que el vacío
maestra_grupos['fisc_imputado'] = fisc.isna()          # True = no tenía scoring real (era 0 o vacío)
maestra_grupos['Score Fiscalización'] = fisc.fillna(2)

# Score Total = Fisc × (Semáforo + 1)
maestra_grupos['Score Total'] = maestra_grupos['Score Fiscalización'] * (maestra_grupos['Score Semáforo'] + 1)

# Copiar el flag de imputado a la maestra (lo lee la celda del JSON)
maestra['fisc_imputado'] = maestra_grupos['fisc_imputado'].values

print('Grupos + SCORE:', maestra_grupos.shape, '| Desglose:', maestra_desglose.shape)
for g in GRUPOS:
    print(' ', g, '->', maestra_grupos[g].value_counts().to_dict())

## Export — Excel final

In [ ]:
OUTPUT_FILE = os.path.join(OUTPUT_DIR, NOMBRE_EXCEL_SALIDA)

# Datos de referencia para enriquecer desglosados
REF_PLANTA = df_pad[['establecimiento_ruca','cuit_titular','razon_social']].drop_duplicates(subset='establecimiento_ruca')

def enriquecer(df_ind):
    """Agrega razon_social y cuit_titular desde el padrón SIOCAL."""
    if 'razon_social' not in df_ind.columns:
        df_ind = df_ind.merge(REF_PLANTA, on='establecimiento_ruca', how='left')
    elif 'cuit_titular' not in df_ind.columns:
        df_ind = df_ind.merge(REF_PLANTA[['establecimiento_ruca','cuit_titular']], on='establecimiento_ruca', how='left')
    return df_ind

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    maestra.to_excel(writer, sheet_name='Tabla Maestra', index=False)
    maestra_grupos.to_excel(writer, sheet_name='Semáforo por Grupo', index=False)
    maestra_desglose.to_excel(writer, sheet_name='Desglose por Grupo', index=False)

    enriquecer(ind01_planta).to_excel(writer, sheet_name='Ind01_DTE_Cerr_Amp', index=False)
    enriquecer(ind03_planta).to_excel(writer, sheet_name='Ind03_Faltante', index=False)
    ind05_detalle.to_excel(writer, sheet_name='Ind05_Corral_vs_Ingresos', index=False)
    enriquecer(ind06_planta).to_excel(writer, sheet_name='Ind06_Emergencia', index=False)
    enriquecer(ind07_planta).to_excel(writer, sheet_name='Ind07_Posdatadas', index=False)
    enriquecer(ind08_planta).to_excel(writer, sheet_name='Ind08_AproTardia', index=False)
    enriquecer(ind09_planta).to_excel(writer, sheet_name='Ind09_Demora', index=False)
    enriquecer(ind10_planta).to_excel(writer, sheet_name='Ind10_Denticion_Unif', index=False)
    enriquecer(ind04_planta).to_excel(writer, sheet_name='Ind04_Coincidencia_Sexo', index=False)
    enriquecer(ind11_planta).to_excel(writer, sheet_name='Ind11_Tipif_Habilitado', index=False)
    ind12_detalle.to_excel(writer, sheet_name='Ind12_Entropia_Tipif', index=False)
    enriquecer(ind13_planta).to_excel(writer, sheet_name='Ind13_Faena_vs_Ingreso', index=False)
    ind15_detalle.to_excel(writer, sheet_name='Ind15_Stock_vs_Camara', index=False)
    enriquecer(ind16_planta).to_excel(writer, sheet_name='Ind16_Faena_vs_Existencias', index=False)
    ind14_detalle.to_excel(writer, sheet_name='Ind14_Romaneo_vs_Cupo', index=False)

    # --- Coloreado de las hojas de semáforo ---
    from openpyxl.styles import PatternFill
    _FILLS = {
        'ROJO':     PatternFill('solid', fgColor='F4C7C3'),
        'AMARILLO': PatternFill('solid', fgColor='FCE8B2'),
        'VERDE':    PatternFill('solid', fgColor='B7E1CD'),
        'VIOLETA':  PatternFill('solid', fgColor='D9B3E6'),
        'GRIS':     PatternFill('solid', fgColor='D9D9D9'),
    }
    for _hoja in ['Semáforo por Grupo', 'Desglose por Grupo']:
        _ws = writer.sheets[_hoja]
        for _row in _ws.iter_rows(min_row=2):
            for _cell in _row:
                _f = _FILLS.get(str(_cell.value).upper())
                if _f: _cell.fill = _f
        _ws.freeze_panes = 'A2'

print(f'✅ Exportado: {OUTPUT_FILE}')


---
## Resumen de convenciones

| Convención | Detalle |
|---|---|
| Clave primaria | `establecimiento_ruca` (Int64) |
| Clave titular | `cuit_titular` (Int64, 11 dígitos sin guiones) |
| Clave usuario | `cuit_usuario_faena` (Int64) |
| Campos numéricos | `normalizar_numerico()` — coerce + round + log de NA |
| DTe | string `'0XXXXXXXXX-X'` |
| Fechas | `datetime64[ns]` |
| Columnas | snake_case sin tildes |

**Pendientes:**
- Confirmar si Faena trae `Cuit Usuario Faena` (simplificaría ind09 y ind13)
- Confirmar si Autorizaciones trae `Cuit Usuario Faena` (simplificaría ind06, 07, 08)
- Confirmar si relación DTe-Autorización es 1 a 1


In [ ]:
# === DIAGNÓSTICO DE PERÍODOS POR INDICADOR ===
print('Indicador | Período | Especie/Base')
print('-' * 60)

for nombre, fecha_ini, fecha_fin, base in [
    ('01 - DTE Cerrados',        fecha_ini_01, fecha_fin_01, 'SIGSA'),
    ('03 - Faltante DTe',        fecha_ini_03, fecha_fin_03, 'Ingresos × SIGSA'),
    ('04 - Coincidencia Sexo',   fecha_ini_04, fecha_fin_04, 'SIGSA × Faena'),
    ('05 - Exceso Corral',       fecha_ini_05, fecha_fin_05, 'Ingresos'),
    ('06 - Emergencia',          fecha_ini_06, fecha_fin_06, 'Autorizaciones'),
    ('07 - Posdatadas',          fecha_ini_07, fecha_fin_07, 'Autorizaciones'),
    ('08 - Aprob Tardías',       fecha_ini_08, fecha_fin_08, 'Autorizaciones'),
    ('09 - Demora Faena',        fecha_ini_09, fecha_fin_09, 'Ingresos × Faena'),
    ('13 - % Faenado',           fecha_ini_13, fecha_fin_13, 'Ingresos × Faena'),
    ('14 - Romaneo vs Cupo',     fecha_ini_14, fecha_fin_14, 'Faena'),
    ('15 - Stock vs Cámara',     fecha_ini_15, fecha_fin_15, 'Existencias'),
    ('16 - Faena sin Exist',     fecha_ini_16, fecha_fin_16, 'Faena × Existencias'),
]:
    print(f'{nombre:<28} {str(fecha_ini.date())} → {str(fecha_fin.date())}  |  {base}')

print()
print('Indicadores sin filtro de fecha (usan toda la base):')
print('  10 - Dentición Uniforme  →  Tipificación completa')
ultima_fecha_tip = df_tip['fecha_faena'].dropna().max().normalize()
print('  11 - Tipificador Habilitado  →  Última fecha disponible:', ultima_fecha_tip.date())
print('  12 - Entropía Tipificación  →  Tipificación completa')

print()
print('Especies en Faena (últimos 30 días ind04/14):')
print(df_fae[df_fae['fecha_faena'] >= fecha_ini_04]['especie'].value_counts().to_string())

# Tamaños Frigoríficos

In [ ]:
# === DISTRIBUCIÓN DE FAENA POR ESTABLECIMIENTO (cabezas por mes) — para redefinir tamaños ===
from IPython.display import display

fae = df_fae   # opcional acotar: fae = df_fae[df_fae['fecha_faena'].between('2026-05-01','2026-06-30')]
print('Período en df_fae:', fae['fecha_faena'].min().date(), '→', fae['fecha_faena'].max().date())

# Nombres EXACTOS de la base (plural, y 'Equidos' no 'Equinos')
ESPECIES_VALIDAS = ['Bovinos', 'Porcinos', 'Ovinos', 'Caprinos', 'Bubalinos', 'Equidos']
_esp = fae['especie'].astype(str).str.strip()
_fuera = sorted(_esp[~_esp.isin(ESPECIES_VALIDAS)].dropna().unique())
if _fuera:
    print('⚠️  Especies descartadas (no válidas):', _fuera)
fae = fae.assign(_especie=_esp)
fae = fae[fae['_especie'].isin(ESPECIES_VALIDAS)]

_na = fae[fae['cantidad_cabezas'].isna()]
if len(_na):
    print('⚠️  Filas sin cantidad_cabezas (cortes sin divisor):', _na['_especie'].value_counts().to_dict())

# cabezas/mes = total del período / meses con faena, por establecimiento y especie
fae = fae.assign(_mes=fae['fecha_faena'].dt.to_period('M'))
por_mes = (fae.groupby(['establecimiento_ruca', '_especie'])
           .agg(cabezas_tot=('cantidad_cabezas', 'sum'),
                meses=('_mes', 'nunique')).reset_index())
por_mes['cabezas'] = por_mes['cabezas_tot'] / por_mes['meses']

faena_ee    = por_mes[['establecimiento_ruca', '_especie', 'cabezas']]
faena_estab = faena_ee.groupby('establecimiento_ruca')['cabezas'].sum()

BANDAS = [0, 50, 100, 250, 500, 1000, 2500, 5000, 10000, 25000, 50000, float('inf')]

def tabla_distribucion(cab, bandas=BANDAS):
    s = cab[cab > 0]
    total = s.sum()
    labels = [f'{int(b0):,}–{("∞" if b1==float("inf") else format(int(b1),","))}'
              for b0, b1 in zip(bandas[:-1], bandas[1:])]
    cats = pd.cut(s, bins=bandas, right=False, labels=labels)
    g = s.groupby(cats, observed=False)
    tab = pd.DataFrame({'Establecimientos': g.size(), 'Cabezas/mes': g.sum().round(0).astype('Int64')})
    tab['% Estab.']      = (tab['Establecimientos'] / tab['Establecimientos'].sum() * 100).round(1)
    tab['% Faena']       = (tab['Cabezas/mes'] / total * 100).round(1)
    tab['% Faena acum.'] = tab['% Faena'][::-1].cumsum()[::-1].round(1)
    tab.index.name = 'Rango cabezas/mes'
    return tab[['Establecimientos','% Estab.','Cabezas/mes','% Faena','% Faena acum.']]

print(f'\n===== GENERAL (todas las especies) — {(faena_estab>0).sum()} estab. =====')
display(tabla_distribucion(faena_estab))

for esp in ESPECIES_VALIDAS:
    sub = faena_ee[faena_ee['_especie'] == esp].set_index('establecimiento_ruca')['cabezas']
    if (sub > 0).sum() == 0:
        continue
    print(f'\n===== {esp} — {(sub>0).sum()} estab. =====')
    display(tabla_distribucion(sub))

In [ ]:
# === Inventario de columnas ORIGINALES de cada base ===
import os

def _cols_originales(path):
    ext = os.path.splitext(str(path))[1].lower()
    try:
        if ext == '.csv':
            # probar separadores conocidos, solo el encabezado
            for sep in (';', ',', '\t'):
                d = pd.read_csv(path, sep=sep, nrows=0, encoding='latin-1')
                if d.shape[1] > 1:
                    return list(d.columns)
            return list(pd.read_csv(path, sep=None, engine='python', nrows=0, encoding='latin-1').columns)
        else:
            try:
                return list(pd.read_excel(path, nrows=0, engine='calamine').columns)
            except Exception:
                return list(pd.read_excel(path, nrows=0).columns)
    except Exception as e:
        return [f'⚠️ error al leer: {e}']

for nombre, path in asignacion.items():
    cols = _cols_originales(path)
    print(f'\n===== {nombre}  ({os.path.basename(str(path))}) — {len(cols)} columnas =====')
    for c in cols:
        print('   ', repr(c))

# Loop mensual → datos_tablero.json

Pegá estas celdas **al final de tu notebook v7**, después de que todas las bases estén cargadas y limpias (Fase 1) y de que exista la `maestra` con atributos (transito, tamaño, region, especies, scoring).

Produce `datos_tablero.json` con el contrato de 4 bloques: `meta`, `plantas`, `valores`, `faena`.
Las celdas viejas de indicadores (single-shot) las podés dejar: siguen alimentando el Excel y sirven para **validar** contra el loop.

## A · Parámetros y helpers del loop

In [ ]:
# MESES ahora viene de la hoja "Meses_Procesar" del Excel de parámetros
# (cargado en la celda "0.1 · Parámetros desde Excel").

import calendar

def dias_del_mes(anio, mes):
    return calendar.monthrange(anio, mes)[1]

def plantas_activas(fae_m, ing_m):
    """Plantas que operaron el mes: faenaron (cabezas>0) o ingresaron hacienda."""
    a  = set(fae_m.loc[fae_m['cantidad_cabezas'] > 0, 'establecimiento_ruca'].dropna())
    a |= set(ing_m['establecimiento_ruca'].dropna())
    return pd.Index(sorted(int(x) for x in a))

# --- Aplicabilidad de tipificación (10, 11, 12): solo bovinos ---
# 'especies' en el padrón viene en SINGULAR ('Bovino, Porcino'); ojo, la faena viene en plural.
PLANTAS_CON_BOVINO = set(
    df_pad.loc[df_pad['especies'].astype(str).str.contains('Bovino', case=False, na=False),
               'establecimiento_ruca'].dropna().astype(int)
)
def ingreso_bovinos_mes(ing_m):
    """Rucas que ingresaron BOVINOS ese mes (especie en plural en Ingresos)."""
    m = ing_m['especie'].astype(str).str.strip().str.contains('Bovino', case=False, na=False)
    return set(ing_m.loc[m, 'establecimiento_ruca'].dropna().astype(int))

# --- Lookups que NO se recortan al mes (se usan como referencia cruzada) ---
# 09 y 13: un DTe ingresado en el mes puede faenarse después → la faena por DTe se mira sobre TODA la base.
FAE_POR_DTE_CAB = df_fae.groupby('dte', as_index=False)['cantidad_cabezas'].sum().rename(columns={'cantidad_cabezas':'cabezas_faenadas'})
FAE_POR_DTE_FIN = df_fae.groupby('dte', as_index=False)['fecha_faena'].max().rename(columns={'fecha_faena':'fecha_faena_final'})

# Capacidades (estáticas, no dependen del mes)
CAP_CORRAL = df_cam[df_cam['tipo_division']=='corral'].groupby('establecimiento_ruca')['capacidad_division'].sum()
CAP_CAMARA = df_cam[df_cam['tipo_division']=='camara'].groupby('establecimiento_ruca')['capacidad_division'].sum()

print('Meses:', MESES)
print('Plantas con bovino en padrón:', len(PLANTAS_CON_BOVINO))

## B · Funciones de indicador (una por mes)

Cada `calc_indXX(recorte_del_mes, activas, ...)` devuelve una **Serie indexada por ruca** con:
`float` (hay dato) · `'nd'` (operó y no declaró) · `np.nan` (no aplica → null en el JSON).

Regla de aplicabilidad:
- **12 no-tipificación:** default `'nd'` para toda planta activa; se pisa con el número donde hay dato.
- **Tipificación (10, 11, 12):** `null` si la planta no tiene bovino en el padrón o no ingresó bovinos el mes; `'nd'` si corresponde y no hay filas; número si hay.

⚠️ Envueltos y listos para validar: **01, 03, 06, 07, 08, 09, 10, 11, 13, 16**.
⚠️ Faltan por confirmar contra tus datos: **04, 05, 15** (los dejé escritos pero revisá la validación) y **12 (entropía) + 14 (cupo)**, que quedaron como TODO por ser los más pesados.

In [ ]:

def _base_nd(serie_valor, activas):
    """Default 'nd' para activas; número donde serie_valor tiene dato."""
    out = pd.Series('nd', index=activas, dtype=object)
    idx = out.index.intersection(serie_valor.index)
    out.loc[idx] = serie_valor.loc[idx].values
    return out

# ---- 01 · % DTe Cerrados/Emitidos (SIGSA · fecha_movimiento) ----
def calc_ind01(sig_m, activas):
    g = sig_m.groupby('establecimiento_ruca').agg(amp=('cantidad_amparada','sum'), cie=('cantidad_cierre','sum'))
    val = (g['cie'] / g['amp'] * 100).round(1)
    return _base_nd(val, activas)

# ---- 03 · % DTe con Faltante (Ingresos·fecha_ingreso × SIGSA·fecha_movimiento) ----
def calc_ind03(sig_m, ing_m, activas):
    sig_dte = sig_m.groupby('dte', as_index=False)['cantidad_amparada'].sum()
    ing_dte = ing_m[['dte','cantidad_ingresada','establecimiento_ruca']].drop_duplicates(subset='dte')
    cr = ing_dte.merge(sig_dte, on='dte', how='inner')
    cr['falt'] = (cr['cantidad_ingresada'] - cr['cantidad_amparada']) < 0
    g = cr.groupby('establecimiento_ruca').agg(tot=('dte','count'), f=('falt','sum'))
    val = (g['f'] / g['tot'] * 100).round(1)
    return _base_nd(val, activas)

# ---- 06 · % Emergencia (Autoriz · fecha_faena) ----
def calc_ind06(aut_m, activas):
    g = aut_m.groupby('establecimiento_ruca').agg(t=('dte','count'), e=('es_emergencia','sum'))
    return _base_nd((g['e']/g['t']*100).round(1), activas)

# ---- 07 · % Posdatadas (Autoriz · fecha_faena) ----
def calc_ind07(aut_m, activas):
    g = aut_m.groupby('establecimiento_ruca').agg(t=('dte','count'), p=('es_posdatada','sum'))
    return _base_nd((g['p']/g['t']*100).round(1), activas)

# ---- 08 · % Aprobaciones Tardías (Autoriz · fecha_faena + fecha_aprobada) ----
def calc_ind08(aut_m, activas):
    d = aut_m[aut_m['autorizacion_fecha_aprobada'].notna()].copy()
    if len(d)==0: return pd.Series('nd', index=activas, dtype=object)
    ff = d['fecha_faena'].dt.normalize(); fa = d['autorizacion_fecha_aprobada'].dt.normalize()
    lim = pd.Timestamp(HORA_LIMITE_APROBACION).time()
    d['tardia'] = ((fa==ff) & (d['autorizacion_fecha_aprobada'].dt.time > lim)) | (fa > ff)
    g = d.groupby('establecimiento_ruca').agg(t=('dte','count'), x=('tardia','sum'))
    return _base_nd((g['x']/g['t']*100).round(1), activas)

# ---- 09 · % Demora Faena >3d (Ingresos·fecha_ingreso × faena TODA) ----
def calc_ind09(ing_m, activas):
    ing_dte = ing_m[['dte','fecha_ingreso','establecimiento_ruca']].drop_duplicates(subset='dte')
    cr = ing_dte.merge(FAE_POR_DTE_FIN, on='dte', how='inner')
    cr['dem'] = (cr['fecha_faena_final'] - cr['fecha_ingreso']).dt.days
    g = cr.groupby('establecimiento_ruca').agg(tot=('dte','count'), d=('dem', lambda x:(x>DEMORA_FAENA_DIAS).sum()))
    return _base_nd((g['d']/g['tot']*100).round(1), activas)

# ---- 13 · % Faenado/Ingresado (Ingresos·fecha_ingreso × faena TODA) ----
def calc_ind13(ing_m, activas):
    ing_dte = ing_m[['dte','cantidad_ingresada','establecimiento_ruca']].drop_duplicates(subset='dte')
    cr = ing_dte.merge(FAE_POR_DTE_CAB, on='dte', how='left')
    cr['cabezas_faenadas'] = cr['cabezas_faenadas'].fillna(0)
    g = cr.groupby('establecimiento_ruca').agg(ing=('cantidad_ingresada','sum'), fae=('cabezas_faenadas','sum'))
    return _base_nd((g['fae']/g['ing']*100).round(1), activas)

# ---- 16 · % Días Faena sin Existencias (Faena·fecha_faena × Existencias·fecha_existencia) ----
def calc_ind16(fae_m, exi_m, activas):
    df = fae_m.groupby('establecimiento_ruca')['fecha_faena'].nunique().rename('df')
    de = exi_m.groupby('establecimiento_ruca')['fecha_existencia'].nunique().rename('de')
    g = pd.concat([df, de], axis=1).fillna(0)
    # Mantener dtype numérico al excluir plantas sin días de faena.
    denominador = g['df'].astype(float).where(g['df'] > 0)
    val = ((g['df']-g['de']).clip(lower=0) / denominador * 100).round(1)
    return _base_nd(val.dropna(), activas)

# ---- 05 · % Días Exceso Corral (Ingresos·fecha_ingreso) — /días del mes ----
def calc_ind05(ing_m, activas, ndias):
    idia = (ing_m.groupby(['establecimiento_ruca', ing_m['fecha_ingreso'].dt.normalize()])['cantidad_ingresada']
            .sum().reset_index(name='ing'))
    idia = idia.merge(CAP_CORRAL.rename('cap'), on='establecimiento_ruca', how='left')
    idia['exc'] = idia['ing'] > idia['cap']
    g = idia.groupby('establecimiento_ruca')['exc'].sum()
    return _base_nd((g / ndias * 100).round(1), activas)

# ---- 15 · % Días Stock > Cámara (Existencias·fecha_existencia) ----
MAP_CORTE_F = {'1/2 Res':1.0,'1/2 Reses':1.0,'1/4 Traseros':0.5,'1/4 Delanteros':0.5}
def calc_ind15(exi_m, activas, ndias):
    d = exi_m.copy()
    d['mr'] = d['cantidad'] * d['corte'].map(MAP_CORTE_F).fillna(1.0)
    ag = d.groupby(['establecimiento_ruca','fecha_existencia']).agg(mr=('mr','sum')).reset_index()
    ag = ag.merge(CAP_CAMARA.rename('cap'), on='establecimiento_ruca', how='left')
    ag['exc'] = ag['mr'] > ag['cap']
    g = ag.groupby('establecimiento_ruca')['exc'].sum()
    return _base_nd((g / ndias * 100).round(1), activas)

# ---- 04 · No Coincidencia de Sexo (SIGSA×Faena por mes) — REVISAR en validación ----
MAP_FAE_SEXO = {'>8D - NO':'M','>8D - TO':'M','>8D - VA':'H','2D - MEJ':'M','2D - NT':'M','2D - VQ':'H','4D - NT':'M','4D - TO':'M','4D - VQ':'H','6D - NO':'M','6D - TO':'M','6D - VA':'H','8D - NO':'M','8D - TO':'M','8D - VA':'H'}
MAP_SIG_SEXO = {'Vaca':'H','Vaquillona':'H','Ternera':'H','Novillo':'M','Novillito':'M','Toro':'M','Torito/MEJ':'M','Ternero':'M','Bueyes':'M'}
def calc_ind04(sig_m, fae_m, activas):
    s = sig_m[['dte','categoria','cantidad_cierre']].copy(); s['sx']=s['categoria'].map(MAP_SIG_SEXO); s=s.dropna(subset=['sx'])
    sp = s.groupby(['dte','sx'])['cantidad_cierre'].sum().unstack(fill_value=0)
    for c in ['H','M']:
        if c not in sp: sp[c]=0
    f = fae_m[['dte','categoria_faena','cantidad_cabezas','establecimiento_ruca']].copy(); f['sx']=f['categoria_faena'].map(MAP_FAE_SEXO); f=f.dropna(subset=['sx'])
    fp = f.groupby(['dte','sx'])['cantidad_cabezas'].sum().unstack(fill_value=0)
    for c in ['H','M']:
        if c not in fp: fp[c]=0
    ruca = f[['dte','establecimiento_ruca']].drop_duplicates('dte')
    cr = sp.merge(fp, on='dte', suffixes=('_s','_f')).merge(ruca, on='dte', how='left')
    cr['nc'] = (cr['H_s']-cr['H_f']).abs() + (cr['M_s']-cr['M_f']).abs()
    cr['tot'] = cr['H_f']+cr['M_f']
    g = cr.groupby('establecimiento_ruca').agg(t=('tot','sum'), nc=('nc','sum'))
    return _base_nd((g['nc']/g['t']*100).round(1), activas)

# ---- 10 · % Dentición Uniforme (Tipificación·fecha_faena) — aplicabilidad bovino ----
# MIN_ANIMALES_TROPA ahora viene de la celda "0.1 · Parámetros desde Excel"
def _aplica_tip(activas, ing_bov):
    """Serie base para tipificación: null si no bovino/no ingresó bovino; 'nd' si corresponde."""
    out = pd.Series(np.nan, index=activas, dtype=object)
    for r in activas:
        if (r in PLANTAS_CON_BOVINO) and (r in ing_bov):
            out.loc[r] = 'nd'
    return out
def calc_ind10(tip_m, activas, ing_bov):
    out = _aplica_tip(activas, ing_bov)
    tr = tip_m.groupby(['establecimiento_ruca','numero_tropa']).agg(
        n=('cantidad_tipificacion','sum'), dd=('denticion','nunique')).reset_index()
    tr = tr[tr['n'] > MIN_ANIMALES_TROPA]
    if len(tr):
        tr['hom'] = tr['dd']==1
        g = tr.groupby('establecimiento_ruca').agg(tot=('hom','count'), h=('hom','sum'))
        val = (g['h']/g['tot']*100).round(1)
        idx = out.index.intersection(val.index); out.loc[idx] = val.loc[idx].values
    return out

# ---- 11 · Tipificador No Habilitado (Tipificación·fecha_faena) — binario, aplicabilidad bovino ----
PLANTAS_HAB_POR_TIP = df_hab[df_hab['habilitado']].groupby('nro_tipificador')['establecimiento_ruca'].apply(set)
def calc_ind11(tip_m, activas, ing_bov):
    out = _aplica_tip(activas, ing_bov)
    d = tip_m.dropna(subset=['numero_tipificador','establecimiento_ruca']).copy()
    if len(d):
        # última fecha de tip del mes por ruca
        uf = d.groupby('establecimiento_ruca')['fecha_faena'].transform('max')
        d = d[d['fecha_faena'].dt.normalize() == uf.dt.normalize()]
        def ok(row):
            pl = PLANTAS_HAB_POR_TIP.get(row['numero_tipificador'])
            return (pl is not None) and (row['establecimiento_ruca'] in pl)
        d['ok'] = d.apply(ok, axis=1)
        g = d.groupby('establecimiento_ruca')['ok'].apply(lambda x: 1 if (~x).any() else 0)
        idx = out.index.intersection(g.index); out.loc[idx] = g.loc[idx].values
    return out

# ---- 12 · Entropía  — TODO (el más sensible a la mensualización) ----
def calc_ind12(tip_m, activas, ing_bov):
    # TODO: envolver la lógica de entropía de la celda 58, parametrizada por mes.
    return _aplica_tip(activas, ing_bov)

# ---- 14 · Cupo de Faena — TODO (base habilitacion_faena, lógica larga celda 62) ----
def calc_ind14(fae_m, activas):
    return pd.Series('nd', index=activas, dtype=object)  # placeholder


## C · Loop mensual → `valores` + `faena`

In [ ]:
valores, faena = {}, {}
for (anio, mes) in MESES:
    ym = f'{anio}-{mes:02d}'; nd = dias_del_mes(anio, mes)
    sig_m = filtrar_mes(df_sig, 'fecha_movimiento', anio, mes)
    ing_m = filtrar_mes(df_ing, 'fecha_ingreso',    anio, mes)
    aut_m = filtrar_mes(df_aut, 'fecha_faena',       anio, mes)
    fae_m = filtrar_mes(df_fae, 'fecha_faena',       anio, mes)
    exi_m = filtrar_mes(df_exi, 'fecha_existencia',  anio, mes)
    tip_m = filtrar_mes(df_tip, 'fecha_faena',       anio, mes)
    act   = plantas_activas(fae_m, ing_m)
    ingb  = ingreso_bovinos_mes(ing_m)

    cols = {
        '01': calc_ind01(sig_m, act),           '03': calc_ind03(sig_m, ing_m, act),
        '04': calc_ind04(sig_m, fae_m, act),    '05': calc_ind05(ing_m, act, nd),
        '06': calc_ind06(aut_m, act),           '07': calc_ind07(aut_m, act),
        '08': calc_ind08(aut_m, act),           '09': calc_ind09(ing_m, act),
        '10': calc_ind10(tip_m, act, ingb),     '11': calc_ind11(tip_m, act, ingb),
        '12': calc_ind12(tip_m, act, ingb),     '13': calc_ind13(ing_m, act),
        '14': calc_ind14(fae_m, act),           '15': calc_ind15(exi_m, act, nd),
        '16': calc_ind16(fae_m, exi_m, act),
    }
    for ruca in act:
        d = valores.setdefault(int(ruca), {}).setdefault(ym, {})
        for cod, serie in cols.items():
            x = serie.get(ruca)
            if pd.isna(x): d[cod] = None            # null = no aplica
            else:          d[cod] = x               # número o 'nd'
    # faena por especie (plural, tal cual la base)
    fpe = (fae_m[fae_m['cantidad_cabezas']>0]
           .groupby(['establecimiento_ruca','especie'])['cantidad_cabezas'].sum().round(1))
    for (ruca, esp), cab in fpe.items():
        faena.setdefault(int(ruca), {}).setdefault(ym, {})[str(esp)] = float(cab)

print('Plantas con al menos un mes activo:', len(valores))
print('Ejemplo:', next(iter(valores.items())))

## D · Armar `meta` + `plantas` y escribir el JSON

In [ ]:

import json

# UMBRALES_META y GRUPOS_META ahora vienen de la celda "0.1 · Parámetros desde Excel":
# son EXACTAMENTE los mismos umbrales que usa el Excel de salida (antes estaban
# duplicados acá con valores que se habían desalineado — ver columna "Nota" en la
# hoja Indicadores_Semaforo del Excel de parámetros).

# atributos por planta desde la maestra
tip_map = tip_ultimo_mes.set_index('establecimiento_ruca').to_dict('index')
plantas = {}
for _, r in maestra.iterrows():
    ruca = int(r['Establecimiento SIOCAL'])
    if ruca not in valores:  # solo plantas con actividad en el período
        continue
    plantas[ruca] = {
        'razon_social': r.get('Razón Social', r.get('razon_social')),
        'cuit': r.get('CUIT', r.get('cuit')),
        'transito': r.get('Tránsito', r.get('transito')),
        'region': r.get('Región', r.get('region')),
        'provincia': r.get('Provincia'),
        'especies': [e.strip() for e in str(r.get('Especies','')).split(',') if e.strip()],
        # ⚠️ tamaño por especie: reemplazá por tus columnas tamaño_bovino, etc. cuando estén en la maestra
        'tamano': r.get('Tamaño', r.get('tamano')),
        'ultima_inspeccion': r.get('Última Inspección', r.get('ultima_inspeccion')),
        'score_claudio': (None if pd.isna(r.get('Scoring Actual', r.get('scoring_actual'))) else r.get('Scoring Actual', r.get('scoring_actual'))),
        'fisc_imputado': bool(r.get('fisc_imputado', False)),
        'es_bovino': ('Bovino' in [e.strip() for e in str(r.get('Especies','')).split(',')]),
        'cabezas_tipificadas_mes': int(tip_map.get(ruca, {}).get('cabezas_tipificadas_mes', 0)),
        'actividad': r.get('Actividad'),
    }

export = {
    'meta': {'meses': [f'{a}-{m:02d}' for a,m in MESES], 'indicadores': UMBRALES_META, 'grupos': GRUPOS_META},
    'plantas': {str(k): v for k,v in plantas.items()},
    'valores': {str(k): v for k,v in valores.items()},
    'faena':   {str(k): v for k,v in faena.items()},
}
ruta = os.path.join(OUTPUT_DIR, NOMBRE_JSON_SALIDA)
with open(ruta, 'w', encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False)
print('✅ JSON escrito:', ruta)
print('Plantas:', len(export['plantas']), '| tamaño JSON (KB):', round(len(json.dumps(export))/1024))

## E · Validación — cada indicador mensual vs. tu cálculo actual

Compara el **último mes** del loop contra tu `indXX_planta` de v7 (que usa la ventana de 30 días). No van a dar idénticos (mes calendario ≠ últimos 30 días), pero el orden de magnitud y las plantas tienen que coincidir. Si alguno se va lejos, ese lo revisamos.

In [ ]:

ULTIMO = f'{MESES[-1][0]}-{MESES[-1][1]:02d}'
def serie_loop(cod):
    return pd.Series({r: v[ULTIMO].get(cod) for r,v in valores.items() if ULTIMO in v})

PARES = {  # cod -> (df actual, columna del valor)
 '01':(ind01_planta,'01_pct_cerrado_amparado'), '03':(ind03_planta,'03_pct_dte_con_faltante'),
 '04':(ind04_planta,'04_pct_no_coinciden_sexo'), '05':(ind05_planta,'05_pct_dias_exceso_corral'),
 '06':(ind06_planta,'06_pct_emergencia_mes_actual'), '07':(ind07_planta,'07_pct_posdatadas'),
 '08':(ind08_planta,'08_pct_aprobacion_tardia'), '09':(ind09_planta,'09_pct_dte_con_demora'),
 '10':(ind10_planta,'10_pct_tropas_denticion_uniforme'), '13':(ind13_planta,'13_pct_faenado_sobre_ingresado'),
 '16':(ind16_planta,'16_pct_faena_sin_existencias'),
}
print(f'{"Ind":4}{"plantas loop":>13}{"plantas v7":>12}{"corr aprox":>12}')
for cod,(dfv,col) in PARES.items():
    sl = pd.to_numeric(serie_loop(cod), errors='coerce').dropna()
    sv = dfv.set_index('establecimiento_ruca')[col]
    comun = sl.index.intersection(sv.index)
    corr = round(sl.loc[comun].corr(sv.loc[comun]),2) if len(comun)>3 else 'NA'
    print(f'{cod:4}{len(sl):>13}{len(sv):>12}{str(corr):>12}')
print("\\nSi corr ~0.8+ el indicador quedó fiel. Los bajos o NA los revisamos uno por uno.")
